In [1]:
# =========================
# STEP 0: SETUP + LOAD DATA
# =========================
import os, random
import numpy as np
import pandas as pd

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

PATH = "/kaggle/input/stroke-dataset/Stroke.csv"
df = pd.read_csv(PATH)

# Target = first column (per dataset spec)
LABEL_COL = df.columns[0]
print("Target column:", LABEL_COL)

# Basic sanity
print("shape:", df.shape)
print("head columns:", list(df.columns[:15]))

y = df[LABEL_COL].astype(int).values
print("y unique:", np.unique(y))
print("pos rate:", y.mean())

# Drop target from features
X = df.drop(columns=[LABEL_COL])

# Duplicates check
dup_rate = df.duplicated().mean()
print("duplicate row rate:", dup_rate)

os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/raw_loaded.csv", index=False)
print("Saved snapshot: outputs/raw_loaded.csv")

Target column: stroke
shape: (4603, 36)
head columns: ['stroke', 'gender', 'age', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'sleep time', 'diabetes', 'hypertension', 'high cholesterol']
y unique: [0 1]
pos rate: 0.07864436237236584
duplicate row rate: 0.0
Saved snapshot: outputs/raw_loaded.csv


In [2]:
# =========================
# STEP 1A: DATA AUDIT SNAPSHOT
# =========================
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

print("N:", len(df), "| Pos rate:", y.mean())

# 1) Missingness table
miss = df.isna().mean().sort_values(ascending=False)
miss_df = miss.reset_index()
miss_df.columns = ["feature", "missing_rate"]
miss_df.to_csv("outputs/tables/missingness.csv", index=False)
print("Saved: outputs/tables/missingness.csv")

# 2) Missingness plot
plt.figure()
plt.plot(np.arange(len(miss_df)), miss_df["missing_rate"].values)
plt.xlabel("Features (sorted)")
plt.ylabel("Missing rate")
plt.title("Missingness profile")
plt.tight_layout()
plt.savefig("outputs/figures/missingness_profile.png", dpi=300)
plt.savefig("outputs/figures/missingness_profile.pdf", bbox_inches="tight")
plt.close()
print("Saved: outputs/figures/missingness_profile.(png/pdf)")

# 3) Duplicate rows + constant columns
dup_rate = df.duplicated().mean()
print("Duplicate row rate:", dup_rate)

nunique = df.nunique(dropna=False)
const_cols = nunique[nunique <= 1].index.tolist()
print("Constant columns:", const_cols[:20], ("..." if len(const_cols) > 20 else ""))
pd.Series(const_cols).to_csv("outputs/tables/constant_columns.csv", index=False)

# 4) Very high-cardinality categoricals (potential IDs)
high_card = []
for c in df.columns:
    if c == LABEL_COL: 
        continue
    if df[c].dtype == "object":
        uniq = df[c].nunique(dropna=False)
        if uniq > 0.5 * len(df):
            high_card.append((c, uniq))
high_card_df = pd.DataFrame(high_card, columns=["feature", "n_unique"]).sort_values("n_unique", ascending=False)
high_card_df.to_csv("outputs/tables/high_cardinality_object_cols.csv", index=False)
print("Saved: outputs/tables/high_cardinality_object_cols.csv")

print("\nTop 10 missingness:")
print(miss_df.head(10).to_string(index=False))

# =========================
# STEP 1B: FEATURE TYPING (CAT vs NUM) + SAVE LISTS
# =========================
import json
import pandas as pd
import numpy as np
import os

os.makedirs("outputs", exist_ok=True)

# Work only on features (exclude target)
feat_df = df.drop(columns=[LABEL_COL]).copy()

cat_cols, num_cols = [], []
for c in feat_df.columns:
    dt = feat_df[c].dtype
    if (dt == "object") or (str(dt).startswith("category")) or (dt == "bool"):
        cat_cols.append(c)
    else:
        num_cols.append(c)

# Coerce numerics (to avoid hidden strings in numeric columns)
for c in num_cols:
    feat_df[c] = pd.to_numeric(feat_df[c], errors="coerce")

# Decide missingness indicators for numeric (we will add in preprocessing)
num_miss_ind_cols = [f"{c}__MISS" for c in num_cols]

feature_spec = {
    "label_col": LABEL_COL,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "num_miss_ind_cols": num_miss_ind_cols,
    "cat_missing_token": "__MISSING__",
    "num_impute": "median",
    "num_standardize": True
}

with open("outputs/feature_spec.json", "w") as f:
    json.dump(feature_spec, f, indent=2)

print("Saved: outputs/feature_spec.json")
print("n_cat:", len(cat_cols), "n_num:", len(num_cols), "total:", len(cat_cols)+len(num_cols))

print("\nSample categorical:", cat_cols[:12])
print("Sample numeric    :", num_cols[:12])

N: 4603 | Pos rate: 0.07864436237236584
Saved: outputs/tables/missingness.csv
Saved: outputs/figures/missingness_profile.(png/pdf)
Duplicate row rate: 0.0
Constant columns: [] 
Saved: outputs/tables/high_cardinality_object_cols.csv

Top 10 missingness:
                 feature  missing_rate
                  stroke           0.0
                  gender           0.0
                     age           0.0
                    Race           0.0
          Marital status           0.0
                alcohol            0.0
                   smoke           0.0
          sleep disorder           0.0
        Health Insurance           0.0
General health condition           0.0
Saved: outputs/feature_spec.json
n_cat: 0 n_num: 35 total: 35

Sample categorical: []
Sample numeric    : ['gender', 'age', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'sleep time', 'diabetes']


In [3]:
# =========================
# STEP 1C: FIX CAT vs NUM (LOW-CARDINALITY RULE)
# =========================
import pandas as pd
import numpy as np
import json

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]

feat_df = df.drop(columns=[LABEL_COL]).copy()

cat_cols, num_cols = [], []

for c in feat_df.columns:
    col = feat_df[c]
    
    # If integer-like AND few unique values → categorical
    nunique = col.nunique(dropna=False)
    
    if nunique <= 10:
        cat_cols.append(c)
    else:
        num_cols.append(c)

print("Corrected typing:")
print("n_cat:", len(cat_cols))
print("n_num:", len(num_cols))

print("\nCategorical features:")
print(cat_cols)

print("\nNumeric features:")
print(num_cols)

# Save corrected spec
feature_spec = {
    "label_col": LABEL_COL,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "cat_missing_token": "__MISSING__",
    "num_impute": "median",
    "num_standardize": True
}

with open("outputs/feature_spec_corrected.json", "w") as f:
    json.dump(feature_spec, f, indent=2)

print("\nSaved: outputs/feature_spec_corrected.json")

Corrected typing:
n_cat: 15
n_num: 20

Categorical features:
['gender', 'age', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'diabetes', 'hypertension', 'high cholesterol', 'Coronary Heart Disease', 'Body Mass Index']

Numeric features:
['sleep time', 'Minutes sedentary activity', 'Waist Circumference', 'Systolic blood pressure', 'Diastolic blood pressure', 'High-density lipoprotein', 'Triglyceride', 'Low-density lipoprotein', 'Fasting Glucose', 'Glycohemoglobin', 'energy', 'protein', 'Carbohydrate', 'Dietary fiber', 'Total fat', 'Total saturated fatty acids', 'Total monounsaturated fatty acids', 'Total polyunsaturated fatty acids', 'Potassium', 'Sodium']

Saved: outputs/feature_spec_corrected.json


In [4]:
# =========================
# STEP 2A: STRATIFIED FOLDS
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import os

SEED = 42
N_SPLITS = 5

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]

y = df[LABEL_COL].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

folds = np.zeros(len(df), dtype=int)
for k, (_, va_idx) in enumerate(skf.split(df, y)):
    folds[va_idx] = k

print("Fold distribution:")
for k in range(N_SPLITS):
    m = folds == k
    print(f"fold {k}: n={m.sum()} pos_rate={y[m].mean():.4f}")

os.makedirs("outputs", exist_ok=True)
np.save("outputs/folds.npy", folds)
print("\nSaved: outputs/folds.npy")

# =========================
# STEP 2B: FOLD-SAFE PREPROCESSING UTILS
# =========================
import numpy as np
import pandas as pd

# Load feature spec you saved earlier
import json
with open("outputs/feature_spec_corrected.json") as f:
    spec = json.load(f)

cat_cols = spec["cat_cols"]
num_cols = spec["num_cols"]
MISSING_TOKEN = "__MISSING__"

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
folds = np.load("outputs/folds.npy")

# ---------- CATEGORICAL ENCODER ----------
def fit_cat_encoders(df_train, cat_cols):
    encoders = {}
    for c in cat_cols:
        vals = df_train[c].astype(str).fillna(MISSING_TOKEN).unique().tolist()
        mapping = {v:i for i,v in enumerate(vals)}
        encoders[c] = mapping
    return encoders

def transform_cat(df_part, cat_cols, encoders):
    Xc = []
    for c in cat_cols:
        col = df_part[c].astype(str).fillna(MISSING_TOKEN)
        mapping = encoders[c]
        Xc.append(col.map(lambda x: mapping.get(x, 0)).values)
    return np.stack(Xc, axis=1)  # shape (N, n_cat)

# ---------- NUMERIC PREPROCESS ----------
def fit_num_stats(df_train, num_cols):
    stats = {}
    for c in num_cols:
        col = pd.to_numeric(df_train[c], errors="coerce")
        med = col.median()
        mean = col.mean()
        std = col.std() if col.std() > 1e-8 else 1.0
        stats[c] = (med, mean, std)
    return stats

def transform_num(df_part, num_cols, stats):
    Xn, Xm = [], []
    for c in num_cols:
        col = pd.to_numeric(df_part[c], errors="coerce")
        med, mean, std = stats[c]
        miss = col.isna().astype(int).values
        col = col.fillna(med)
        col = (col - mean) / std
        Xn.append(col.values)
        Xm.append(miss)
    return np.stack(Xn,1), np.stack(Xm,1)  # num, missing-indicators

print("Preprocessing utilities ready.")

Fold distribution:
fold 0: n=921 pos_rate=0.0782
fold 1: n=921 pos_rate=0.0793
fold 2: n=921 pos_rate=0.0793
fold 3: n=920 pos_rate=0.0783
fold 4: n=920 pos_rate=0.0783

Saved: outputs/folds.npy
Preprocessing utilities ready.


In [5]:
# =========================
# STEP 3A: BUILD ONE-FOLD ARRAYS (NO LEAKAGE)
# =========================
import numpy as np
import pandas as pd
import json

FOLD = 0  # start with fold 0

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
y_all = df[LABEL_COL].astype(int).values
folds = np.load("outputs/folds.npy")

with open("outputs/feature_spec_corrected.json") as f:
    spec = json.load(f)
cat_cols = spec["cat_cols"]
num_cols = spec["num_cols"]

tr_idx = np.where(folds != FOLD)[0]
va_idx = np.where(folds == FOLD)[0]

df_tr = df.iloc[tr_idx].reset_index(drop=True)
df_va = df.iloc[va_idx].reset_index(drop=True)

# Fit on TRAIN only
enc = fit_cat_encoders(df_tr, cat_cols)
num_stats = fit_num_stats(df_tr, num_cols)

# Transform train/val
Xc_tr = transform_cat(df_tr, cat_cols, enc).astype(np.int64)
Xc_va = transform_cat(df_va, cat_cols, enc).astype(np.int64)

Xn_tr, Xm_tr = transform_num(df_tr, num_cols, num_stats)
Xn_va, Xm_va = transform_num(df_va, num_cols, num_stats)

Xn_tr = Xn_tr.astype(np.float32); Xm_tr = Xm_tr.astype(np.float32)
Xn_va = Xn_va.astype(np.float32); Xm_va = Xm_va.astype(np.float32)

y_tr = df_tr[LABEL_COL].astype(int).values.astype(np.int64)
y_va = df_va[LABEL_COL].astype(int).values.astype(np.int64)

print("Train shapes:", Xc_tr.shape, Xn_tr.shape, Xm_tr.shape, y_tr.shape, "pos:", y_tr.mean())
print("Val   shapes:", Xc_va.shape, Xn_va.shape, Xm_va.shape, y_va.shape, "pos:", y_va.mean())

# Cardinalities (for embeddings)
cat_cardinalities = [df_tr[c].astype(str).fillna("__MISSING__").nunique() for c in cat_cols]
print("Cat cardinalities:", cat_cardinalities)
print("n_cat:", len(cat_cols), "n_num:", len(num_cols))

# =========================
# STEP 3B: TORCH DATASET + DATALOADERS
# =========================
import torch
from torch.utils.data import Dataset, DataLoader

class TabDataset(Dataset):
    def __init__(self, Xc, Xn, Xm, y):
        self.Xc = torch.from_numpy(Xc)              # int64
        self.Xn = torch.from_numpy(Xn)              # float32
        self.Xm = torch.from_numpy(Xm)              # float32
        self.y  = torch.from_numpy(y).float()       # float32 for BCE
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xn[i], self.Xm[i], self.y[i]

BATCH_SIZE = 256

ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)

dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

# quick batch check
xb = next(iter(dl_tr))
print("Batch shapes:",
      xb[0].shape,  # Xc
      xb[1].shape,  # Xn
      xb[2].shape,  # Xm
      xb[3].shape)  # y
print("Device ready.")

Train shapes: (3682, 15) (3682, 20) (3682, 20) (3682,) pos: 0.07876154263986963
Val   shapes: (921, 15) (921, 20) (921, 20) (921,) pos: 0.0781758957654723
Cat cardinalities: [2, 3, 5, 6, 2, 2, 2, 2, 5, 3, 2, 2, 2, 2, 4]
n_cat: 15 n_num: 20
Batch shapes: torch.Size([256, 15]) torch.Size([256, 20]) torch.Size([256, 20]) torch.Size([256])
Device ready.


In [6]:
# =========================
# STEP 4A: TABULAR TRANSFORMER MODEL (LOGITS OUTPUT)
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(embed_dim=d, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4*d, d),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: (B, T, d)
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

class TabularTransformer(nn.Module):
    def __init__(self, cat_cardinalities, n_num, d=64, depth=3, heads=4, dropout=0.1):
        super().__init__()
        self.n_cat = len(cat_cardinalities)
        self.n_num = n_num
        self.d = d

        # Categorical embeddings: one embedding table per categorical feature
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(card, d) for card in cat_cardinalities
        ])

        # Numeric tokenization: each numeric feature gets a learned (w,b) to map scalar->d
        self.num_W = nn.Parameter(torch.randn(n_num, d) * 0.02)
        self.num_b = nn.Parameter(torch.zeros(n_num, d))

        # Missing indicator embedding for numeric features (optional but helpful)
        self.miss_W = nn.Parameter(torch.randn(n_num, d) * 0.02)

        # Positional/type embedding (optional but stabilizes)
        self.type_embed = nn.Embedding(self.n_cat + self.n_num, d)

        self.blocks = nn.ModuleList([TransformerBlock(d, heads, dropout) for _ in range(depth)])
        self.ln = nn.LayerNorm(d)

        # Pool (CLS-free): mean pool tokens then MLP head
        self.head = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1)  # logits
        )

    def forward(self, x_cat, x_num, x_miss):
        # x_cat: (B, n_cat) int64
        # x_num: (B, n_num) float32
        # x_miss:(B, n_num) float32 {0,1}

        B = x_num.size(0)

        # ---- cat tokens ----
        cat_tokens = []
        for j, emb in enumerate(self.cat_embeds):
            cat_tokens.append(emb(x_cat[:, j]))  # (B, d)
        if len(cat_tokens) > 0:
            cat_tokens = torch.stack(cat_tokens, dim=1)  # (B, n_cat, d)
        else:
            cat_tokens = torch.zeros(B, 0, self.d, device=x_num.device)

        # ---- num tokens ----
        # scalar -> token: x * W + b, plus missing embedding
        # (B, n_num, 1) * (n_num, d) -> (B, n_num, d)
        num_tokens = x_num.unsqueeze(-1) * self.num_W.unsqueeze(0) + self.num_b.unsqueeze(0)
        num_tokens = num_tokens + x_miss.unsqueeze(-1) * self.miss_W.unsqueeze(0)

        # ---- concatenate tokens ----
        x = torch.cat([cat_tokens, num_tokens], dim=1)  # (B, T, d), T=n_cat+n_num

        # add type/position embeddings
        T = x.size(1)
        type_ids = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        x = x + self.type_embed(type_ids)

        # transformer
        for blk in self.blocks:
            x = blk(x)

        x = self.ln(x)

        # mean pool tokens
        pooled = x.mean(dim=1)  # (B, d)

        logits = self.head(pooled).squeeze(1)  # (B,)
        return logits

# =========================
# STEP 4B: MODEL SANITY CHECK (FORWARD + LOSS)
# =========================
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = TabularTransformer(
    cat_cardinalities=cat_cardinalities,
    n_num=len(num_cols),
    d=64,
    depth=3,
    heads=4,
    dropout=0.1
).to(device)

# One batch
x_cat_b, x_num_b, x_miss_b, y_b = next(iter(dl_tr))
x_cat_b = x_cat_b.to(device)
x_num_b = x_num_b.to(device)
x_miss_b = x_miss_b.to(device)
y_b = y_b.to(device)

logits = model(x_cat_b, x_num_b, x_miss_b)
print("logits shape:", logits.shape, "logits stats:", logits.min().item(), logits.max().item())

crit = nn.BCEWithLogitsLoss()
loss = crit(logits, y_b)
print("loss:", loss.item())

device: cpu
logits shape: torch.Size([256]) logits stats: -0.06288117170333862 0.06894201040267944
loss: 0.6974148750305176


In [7]:
# =========================
# STEP 5A: METRICS + EVAL UTILS
# =========================
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def ece_binary(y_true, p_pred, n_bins=15):
    # Expected Calibration Error for binary
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

@torch.no_grad()
def predict_logits(model, dl, device):
    model.eval()
    all_logits, all_y = [], []
    for x_cat, x_num, x_miss, y in dl:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)
        x_miss = x_miss.to(device)
        logits = model(x_cat, x_num, x_miss)
        all_logits.append(logits.detach().cpu().numpy())
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

def compute_metrics_from_logits(y_true, logits):
    p = sigmoid_np(logits)
    y_true = y_true.astype(int)

    out = {}
    out["PR_AUC"] = float(average_precision_score(y_true, p))
    out["ROC_AUC"] = float(roc_auc_score(y_true, p))
    out["Brier"]  = float(brier_score_loss(y_true, p))
    out["ECE"]    = float(ece_binary(y_true, p, n_bins=15))
    out["NLL"]    = float(log_loss(y_true, np.clip(p, 1e-6, 1-1e-6)))
    out["p_mean"] = float(p.mean())
    return out

# =========================
# STEP 5B: TRAIN ONE FOLD (NO ENSEMBLE YET)
# =========================
import torch
import torch.nn as nn
import pandas as pd
import os

os.makedirs("outputs/checkpoints", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

# --- Training config (CPU-safe) ---
EPOCHS = 25
LR = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 5

# Imbalance-aware loss (simple pos_weight)
pos = y_tr.mean()
neg = 1.0 - pos
pos_weight = torch.tensor([neg / max(pos, 1e-8)], device=device)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model = TabularTransformer(
    cat_cardinalities=cat_cardinalities,
    n_num=len(num_cols),
    d=64,
    depth=3,
    heads=4,
    dropout=0.1
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_pr = -1.0
best_path = f"outputs/checkpoints/fold{FOLD}_best.pt"
bad = 0
history = []

for ep in range(EPOCHS):
    model.train()
    total_loss = 0.0
    n = 0

    for x_cat, x_num, x_miss, yb in dl_tr:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)
        x_miss = x_miss.to(device)
        yb = yb.to(device)

        opt.zero_grad(set_to_none=True)
        logits = model(x_cat, x_num, x_miss)
        loss = crit(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        bs = yb.size(0)
        total_loss += loss.item() * bs
        n += bs

    tr_loss = total_loss / max(n, 1)

    # Eval
    va_logits, va_y = predict_logits(model, dl_va, device)
    m = compute_metrics_from_logits(va_y, va_logits)

    row = {"epoch": ep, "train_loss": tr_loss, **m}
    history.append(row)

    print(f"ep {ep:02d} loss {tr_loss:.4f} | "
          f"PR {m['PR_AUC']:.4f} ROC {m['ROC_AUC']:.4f} "
          f"Brier {m['Brier']:.4f} ECE {m['ECE']:.4f}")

    # Early stopping on PR-AUC
    if m["PR_AUC"] > best_pr + 1e-5:
        best_pr = m["PR_AUC"]
        bad = 0
        torch.save({"model": model.state_dict(), "spec": {"d":64,"depth":3,"heads":4}}, best_path)
    else:
        bad += 1
        if bad >= PATIENCE:
            print("Early stopping.")
            break

hist_df = pd.DataFrame(history)
hist_df.to_csv(f"outputs/tables/fold{FOLD}_train_history.csv", index=False)
print("Saved:", f"outputs/tables/fold{FOLD}_train_history.csv")
print("Best checkpoint:", best_path, "best PR:", best_pr)

ep 00 loss 1.2765 | PR 0.1756 ROC 0.7088 Brier 0.2401 ECE 0.4125
ep 01 loss 1.2430 | PR 0.1907 ROC 0.7215 Brier 0.3646 ECE 0.5430
ep 02 loss 1.2346 | PR 0.1913 ROC 0.7216 Brier 0.3518 ECE 0.5274
ep 03 loss 1.1834 | PR 0.2006 ROC 0.7272 Brier 0.2356 ECE 0.3969
ep 04 loss 1.1435 | PR 0.2007 ROC 0.7270 Brier 0.2000 ECE 0.3310
ep 05 loss 1.1193 | PR 0.2092 ROC 0.7246 Brier 0.1966 ECE 0.3212
ep 06 loss 1.1322 | PR 0.2045 ROC 0.7277 Brier 0.1898 ECE 0.3207
ep 07 loss 1.1186 | PR 0.2099 ROC 0.7267 Brier 0.2410 ECE 0.3869
ep 08 loss 1.1094 | PR 0.2017 ROC 0.7228 Brier 0.2568 ECE 0.4002
ep 09 loss 1.1068 | PR 0.1933 ROC 0.7237 Brier 0.2384 ECE 0.3799
ep 10 loss 1.0916 | PR 0.1948 ROC 0.7284 Brier 0.1741 ECE 0.2866
ep 11 loss 1.0888 | PR 0.1900 ROC 0.7266 Brier 0.2121 ECE 0.3266
ep 12 loss 1.0873 | PR 0.1905 ROC 0.7237 Brier 0.2161 ECE 0.3385
Early stopping.
Saved: outputs/tables/fold0_train_history.csv
Best checkpoint: outputs/checkpoints/fold0_best.pt best PR: 0.20986976848017208


In [8]:
# =========================
# STEP 6A: LOAD BEST + PREDICT OOF (FOLD)
# =========================
import torch
import numpy as np
import os

best_path = f"outputs/checkpoints/fold{FOLD}_best.pt"
ckpt = torch.load(best_path, map_location=device)

model = TabularTransformer(
    cat_cardinalities=cat_cardinalities,
    n_num=len(num_cols),
    d=64,
    depth=3,
    heads=4,
    dropout=0.1
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

va_logits, va_y = predict_logits(model, dl_va, device)
va_p = 1.0 / (1.0 + np.exp(-va_logits))

print("Val logits stats:", va_logits.min(), va_logits.max(), va_logits.mean())
print("Val prob stats  :", va_p.min(), va_p.max(), va_p.mean())
print("Val pos rate    :", va_y.mean())

os.makedirs("outputs/oof", exist_ok=True)
np.save(f"outputs/oof/fold{FOLD}_va_idx.npy", va_idx)
np.save(f"outputs/oof/fold{FOLD}_va_logits.npy", va_logits)
np.save(f"outputs/oof/fold{FOLD}_va_prob_raw.npy", va_p)

print("Saved raw OOF for fold", FOLD, "to outputs/oof/")

# =========================
# STEP 6B: PRIOR CORRECTION (FOLD) + SAVE FINAL PROBS
# =========================
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score

def prior_correct_probs(p, pi_true, pi_train, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    logit = np.log(p/(1-p))
    delta = np.log(pi_true/(1-pi_true)) - np.log(pi_train/(1-pi_train))
    return 1.0 / (1.0 + np.exp(-(logit + delta)))

# True prevalence on TRAIN fold (natural prevalence)
pi_true = y_tr.mean()

# Training prior:
# For this run we did NOT use balanced sampling, only pos_weight.
# So set pi_train = pi_true (no shift) -> correction becomes identity.
# Later, when we use balanced sampling, set pi_train = 0.5 (or your pos_frac).
pi_train = pi_true

va_p_corr = prior_correct_probs(va_p, pi_true=pi_true, pi_train=pi_train)

print("Raw prob mean :", va_p.mean(), "Brier:", brier_score_loss(va_y, va_p))
print("Corr prob mean:", va_p_corr.mean(), "Brier:", brier_score_loss(va_y, va_p_corr))
print("PR:", average_precision_score(va_y, va_p_corr), "ROC:", roc_auc_score(va_y, va_p_corr))

np.save(f"outputs/oof/fold{FOLD}_va_prob_final.npy", va_p_corr)
print("Saved: outputs/oof/fold{FOLD}_va_prob_final.npy")

Val logits stats: -2.3552837 1.4090904 -0.19971213
Val prob stats  : 0.086646706 0.8036224 0.46469563
Val pos rate    : 0.078175895
Saved raw OOF for fold 0 to outputs/oof/
Raw prob mean : 0.46469563 Brier: 0.24099002457198257
Corr prob mean: 0.4646956283438009 Brier: 0.2409900249259793
PR: 0.20986976848017208 ROC: 0.7266719015835624
Saved: outputs/oof/fold{FOLD}_va_prob_final.npy


In [9]:
# =========================
# STEP 6C: TEMPERATURE SCALING (FOLD)
# =========================
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def fit_temperature(logits, y, iters=2000, lr=0.01):
    # minimize NLL of sigmoid(logits / T) over T>0
    # simple 1D gradient descent in log-space for stability
    y = y.astype(np.float64)
    z = logits.astype(np.float64)

    t = 0.0  # logT
    for _ in range(iters):
        T = np.exp(t)

        p = sigmoid(z / T)
        p = np.clip(p, 1e-8, 1-1e-8)

        # NLL
        # d/dt: chain through T=exp(t)
        # gradient derived from logistic NLL
        # dNLL/dT = sum( (p - y) * (-z / T^2) ) / N
        # dNLL/dt = dNLL/dT * dT/dt = dNLL/dT * T
        dNLL_dT = np.mean((p - y) * (-z / (T*T)))
        grad = dNLL_dT * T

        t -= lr * grad

    return float(np.exp(t))

T = fit_temperature(va_logits, va_y, iters=2000, lr=0.05)
va_p_temp = sigmoid(va_logits / T)

print("Learned T:", T)
print("Temp prob mean:", va_p_temp.mean(), "min/max:", va_p_temp.min(), va_p_temp.max())

# =========================
# STEP 6D: METRICS AFTER TEMPERATURE + SAVE
# =========================
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score, log_loss

def ece_binary(y_true, p_pred, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

raw = va_p
cal = va_p_temp

print("RAW : mean", raw.mean(),
      "Brier", brier_score_loss(va_y, raw),
      "ECE", ece_binary(va_y, raw),
      "NLL", log_loss(va_y, np.clip(raw, 1e-6, 1-1e-6)),
      "PR", average_precision_score(va_y, raw),
      "ROC", roc_auc_score(va_y, raw))

print("TEMP: mean", cal.mean(),
      "Brier", brier_score_loss(va_y, cal),
      "ECE", ece_binary(va_y, cal),
      "NLL", log_loss(va_y, np.clip(cal, 1e-6, 1-1e-6)),
      "PR", average_precision_score(va_y, cal),
      "ROC", roc_auc_score(va_y, cal))

np.save(f"outputs/oof/fold{FOLD}_va_prob_temp.npy", cal)
np.save(f"outputs/oof/fold{FOLD}_T.npy", np.array([T], dtype=np.float32))
print("Saved: outputs/oof/fold{FOLD}_va_prob_temp.npy and fold{FOLD}_T.npy")

Learned T: 1.6418745082442354
Temp prob mean: 0.47359106 min/max: 0.19239712 0.70228875
RAW : mean 0.46469563 Brier 0.24099002457198257 ECE 0.3869461766428979 NLL 0.6666273169564867 PR 0.20986976848017208 ROC 0.7266719015835624
TEMP: mean 0.47359106 Brier 0.23210061885079472 ECE 0.3954151576136145 NLL 0.6529164720814922 PR 0.20986976848017208 ROC 0.7266719015835624
Saved: outputs/oof/fold{FOLD}_va_prob_temp.npy and fold{FOLD}_T.npy


In [10]:
# =========================
# STEP 6E: PLATT SCALING (LOGIT -> CALIBRATED PROB)
# =========================
import numpy as np
from sklearn.linear_model import LogisticRegression

# va_logits, va_y already computed
Z = va_logits.reshape(-1, 1)
cal = LogisticRegression(solver="lbfgs", max_iter=1000)
cal.fit(Z, va_y)

va_p_platt = cal.predict_proba(Z)[:, 1]

print("Platt params: a (coef) =", cal.coef_[0][0], "b (intercept) =", cal.intercept_[0])
print("Platt prob mean:", va_p_platt.mean(), "min/max:", va_p_platt.min(), va_p_platt.max())
# =========================
# STEP 6F: METRICS AFTER PLATT + SAVE
# =========================
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score, log_loss

def ece_binary(y_true, p_pred, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

raw = va_p
platt = va_p_platt

print("RAW  : mean", raw.mean(),
      "Brier", brier_score_loss(va_y, raw),
      "ECE", ece_binary(va_y, raw),
      "NLL", log_loss(va_y, np.clip(raw, 1e-6, 1-1e-6)),
      "PR", average_precision_score(va_y, raw),
      "ROC", roc_auc_score(va_y, raw))

print("PLATT: mean", platt.mean(),
      "Brier", brier_score_loss(va_y, platt),
      "ECE", ece_binary(va_y, platt),
      "NLL", log_loss(va_y, np.clip(platt, 1e-6, 1-1e-6)),
      "PR", average_precision_score(va_y, platt),
      "ROC", roc_auc_score(va_y, platt))

import os
os.makedirs("outputs/oof", exist_ok=True)
np.save(f"outputs/oof/fold{FOLD}_va_prob_platt.npy", platt)
np.save(f"outputs/oof/fold{FOLD}_platt_coef.npy", np.array([cal.coef_[0][0], cal.intercept_[0]], dtype=np.float32))
print("Saved: outputs/oof/fold{FOLD}_va_prob_platt.npy and fold{FOLD}_platt_coef.npy")

Platt params: a (coef) = 1.0542281177237876 b (intercept) = -2.622162370756443
Platt prob mean: 0.07817376320875358 min/max: 0.006028749272426695 0.24293418087400512
RAW  : mean 0.46469563 Brier 0.24099002457198257 ECE 0.3869461766428979 NLL 0.6666273169564867 PR 0.20986976848017208 ROC 0.7266719015835624
PLATT: mean 0.07817376320875358 Brier 0.06824487449653661 ECE 0.007897513496672202 NLL 0.24934745028828917 PR 0.20986976848017208 ROC 0.7266719015835624
Saved: outputs/oof/fold{FOLD}_va_prob_platt.npy and fold{FOLD}_platt_coef.npy


In [11]:
# =========================
# STEP 7A: ALL-FOLDS OOF (TRAIN + PLATT-CAL PROBS)
# =========================
import numpy as np
import pandas as pd
import json, os
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss

# ---- Config ----
SEED = 42
N_SPLITS = 5
EPOCHS = 25
LR = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 5
BATCH_SIZE = 256

os.makedirs("outputs/checkpoints", exist_ok=True)
os.makedirs("outputs/oof", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

# ---- Load data/spec/folds ----
df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
y_all = df[LABEL_COL].astype(int).values
folds = np.load("outputs/folds.npy")

with open("outputs/feature_spec_corrected.json") as f:
    spec = json.load(f)
cat_cols = spec["cat_cols"]
num_cols = spec["num_cols"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- Helpers: reuse your preprocess + dataset from earlier steps ----
# fit_cat_encoders, transform_cat, fit_num_stats, transform_num
# TabDataset, TabularTransformer, predict_logits already defined

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def ece_binary(y_true, p_pred, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def metrics(y, p):
    return {
        "PR_AUC": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "ECE": float(ece_binary(y, p, n_bins=15)),
        "NLL": float(log_loss(y, np.clip(p, 1e-6, 1-1e-6))),
        "p_mean": float(p.mean()),
    }

# Allocate OOF storage
p_oof = np.zeros(len(df), dtype=np.float32)
logits_oof = np.zeros(len(df), dtype=np.float32)
platt_params = {}

fold_rows = []

for FOLD in range(N_SPLITS):
    tr_idx = np.where(folds != FOLD)[0]
    va_idx = np.where(folds == FOLD)[0]

    df_tr = df.iloc[tr_idx].reset_index(drop=True)
    df_va = df.iloc[va_idx].reset_index(drop=True)

    # fit preprocess on TRAIN
    enc = fit_cat_encoders(df_tr, cat_cols)
    num_stats = fit_num_stats(df_tr, num_cols)

    Xc_tr = transform_cat(df_tr, cat_cols, enc).astype(np.int64)
    Xc_va = transform_cat(df_va, cat_cols, enc).astype(np.int64)

    Xn_tr, Xm_tr = transform_num(df_tr, num_cols, num_stats)
    Xn_va, Xm_va = transform_num(df_va, num_cols, num_stats)

    Xn_tr = Xn_tr.astype(np.float32); Xm_tr = Xm_tr.astype(np.float32)
    Xn_va = Xn_va.astype(np.float32); Xm_va = Xm_va.astype(np.float32)

    y_tr = df_tr[LABEL_COL].astype(int).values.astype(np.int64)
    y_va = df_va[LABEL_COL].astype(int).values.astype(np.int64)

    # cat cardinalities from TRAIN
    cat_cardinalities = [df_tr[c].astype(str).fillna("__MISSING__").nunique() for c in cat_cols]

    ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
    ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)

    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # loss with pos_weight
    pos = y_tr.mean()
    neg = 1.0 - pos
    pos_weight = torch.tensor([neg / max(pos, 1e-8)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = TabularTransformer(
        cat_cardinalities=cat_cardinalities,
        n_num=len(num_cols),
        d=64, depth=3, heads=4, dropout=0.1
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_pr = -1.0
    best_path = f"outputs/checkpoints/fold{FOLD}_best.pt"
    bad = 0

    for ep in range(EPOCHS):
        model.train()
        for x_cat, x_num, x_miss, yb in dl_tr:
            x_cat = x_cat.to(device)
            x_num = x_num.to(device)
            x_miss = x_miss.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            lg = model(x_cat, x_num, x_miss)
            loss = crit(lg, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # eval PR for early stopping
        va_logits, va_y2 = predict_logits(model, dl_va, device)
        va_p_raw = sigmoid_np(va_logits)
        pr = average_precision_score(va_y2.astype(int), va_p_raw)

        if pr > best_pr + 1e-5:
            best_pr = pr
            bad = 0
            torch.save({"model": model.state_dict(), "cat_cardinalities": cat_cardinalities}, best_path)
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    # load best and get final logits
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    va_logits, va_y2 = predict_logits(model, dl_va, device)

    # PLATT on validation logits (slope + intercept)
    Z = va_logits.reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs", max_iter=1000)
    cal.fit(Z, va_y2.astype(int))
    va_p_platt = cal.predict_proba(Z)[:, 1]

    # store aligned OOF
    logits_oof[va_idx] = va_logits.astype(np.float32)
    p_oof[va_idx] = va_p_platt.astype(np.float32)

    platt_params[FOLD] = (float(cal.coef_[0][0]), float(cal.intercept_[0]))

    m = metrics(va_y2.astype(int), va_p_platt)
    fold_rows.append({"fold": FOLD, "best_pr_raw": float(best_pr), **m})
    print(f"[fold {FOLD}] PR {m['PR_AUC']:.4f} ROC {m['ROC_AUC']:.4f} Brier {m['Brier']:.4f} ECE {m['ECE']:.4f} meanP {m['p_mean']:.4f}")

# save fold table + oof arrays
pd.DataFrame(fold_rows).to_csv("outputs/tables/fold_metrics_platt.csv", index=False)
np.save("outputs/oof/oof_logits.npy", logits_oof)
np.save("outputs/oof/oof_prob_platt.npy", p_oof)

with open("outputs/oof/platt_params_by_fold.json", "w") as f:
    import json
    json.dump(platt_params, f, indent=2)

print("Saved: outputs/tables/fold_metrics_platt.csv")
print("Saved: outputs/oof/oof_logits.npy")
print("Saved: outputs/oof/oof_prob_platt.npy")
print("Saved: outputs/oof/platt_params_by_fold.json")

# =========================
# STEP 7B: OVERALL OOF METRICS (PLATT-CAL)
# =========================
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

p_oof = np.load("outputs/oof/oof_prob_platt.npy")
y = y_all.astype(int)

def ece_binary(y_true, p_pred, n_bins=15):
    import numpy as np
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

print("==== OOF OVERALL (PLATT-CAL) ====")
print("PR-AUC:", average_precision_score(y, p_oof))
print("ROC-AUC:", roc_auc_score(y, p_oof))
print("Brier:", brier_score_loss(y, p_oof))
print("ECE:", ece_binary(y, p_oof))
print("NLL:", log_loss(y, np.clip(p_oof, 1e-6, 1-1e-6)))
print("mean(p):", p_oof.mean(), "pos rate:", y.mean())

device: cpu
[fold 0] PR 0.2232 ROC 0.7190 Brier 0.0685 ECE 0.0065 meanP 0.0782
[fold 1] PR 0.1790 ROC 0.6846 Brier 0.0706 ECE 0.0053 meanP 0.0793
[fold 2] PR 0.1738 ROC 0.6917 Brier 0.0726 ECE 0.0000 meanP 0.0793
[fold 3] PR 0.1792 ROC 0.7043 Brier 0.0689 ECE 0.0105 meanP 0.0782
[fold 4] PR 0.2143 ROC 0.7360 Brier 0.0681 ECE 0.0166 meanP 0.0782
Saved: outputs/tables/fold_metrics_platt.csv
Saved: outputs/oof/oof_logits.npy
Saved: outputs/oof/oof_prob_platt.npy
Saved: outputs/oof/platt_params_by_fold.json
==== OOF OVERALL (PLATT-CAL) ====
PR-AUC: 0.1710017430422955
ROC-AUC: 0.6912877578909383
Brier: 0.0697626503969153
ECE: 0.0062858011932080925
NLL: 0.2581964181357921
mean(p): 0.07863146 pos rate: 0.07864436237236584


In [12]:
# =========================
# STEP 8A: ENSEMBLE (ONE FOLD, M=3)
# =========================
import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

FOLD = 0
M = 3  # small test ensemble

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
y_all = df[LABEL_COL].astype(int).values
folds = np.load("outputs/folds.npy")

with open("outputs/feature_spec_corrected.json") as f:
    spec = json.load(f)
cat_cols = spec["cat_cols"]
num_cols = spec["num_cols"]

tr_idx = np.where(folds != FOLD)[0]
va_idx = np.where(folds == FOLD)[0]

df_tr = df.iloc[tr_idx].reset_index(drop=True)
df_va = df.iloc[va_idx].reset_index(drop=True)

# fit preprocess on TRAIN
enc = fit_cat_encoders(df_tr, cat_cols)
num_stats = fit_num_stats(df_tr, num_cols)

Xc_tr = transform_cat(df_tr, cat_cols, enc).astype(np.int64)
Xc_va = transform_cat(df_va, cat_cols, enc).astype(np.int64)

Xn_tr, Xm_tr = transform_num(df_tr, num_cols, num_stats)
Xn_va, Xm_va = transform_num(df_va, num_cols, num_stats)

Xn_tr = Xn_tr.astype(np.float32); Xm_tr = Xm_tr.astype(np.float32)
Xn_va = Xn_va.astype(np.float32); Xm_va = Xm_va.astype(np.float32)

y_tr = df_tr[LABEL_COL].astype(int).values.astype(np.int64)
y_va = df_va[LABEL_COL].astype(int).values.astype(np.int64)

cat_cardinalities = [df_tr[c].astype(str).fillna("__MISSING__").nunique() for c in cat_cols]

ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)

dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True)
dl_va = torch.utils.data.DataLoader(ds_va, batch_size=256, shuffle=False)

member_probs = []

for m in range(M):
    print(f"\n=== Training member {m+1}/{M} ===")

    model = TabularTransformer(
        cat_cardinalities=cat_cardinalities,
        n_num=len(num_cols),
        d=64, depth=3, heads=4, dropout=0.1
    ).to(device)

    pos = y_tr.mean()
    neg = 1.0 - pos
    pos_weight = torch.tensor([neg / max(pos, 1e-8)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    # small training (reuse early stopping logic simplified)
    best_pr = -1.0
    best_state = None
    bad = 0

    for ep in range(25):
        model.train()
        for x_cat, x_num, x_miss, yb in dl_tr:
            x_cat = x_cat.to(device)
            x_num = x_num.to(device)
            x_miss = x_miss.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            lg = model(x_cat, x_num, x_miss)
            loss = crit(lg, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        va_logits, va_y2 = predict_logits(model, dl_va, device)
        va_p_raw = 1/(1+np.exp(-va_logits))
        pr = average_precision_score(va_y2, va_p_raw)

        if pr > best_pr:
            best_pr = pr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= 5:
                break

    # load best state
    model.load_state_dict(best_state)

    # logits → Platt calibration
    va_logits, va_y2 = predict_logits(model, dl_va, device)
    Z = va_logits.reshape(-1,1)
    cal = LogisticRegression(max_iter=1000)
    cal.fit(Z, va_y2)
    p_platt = cal.predict_proba(Z)[:,1]

    member_probs.append(p_platt)

member_probs = np.stack(member_probs, axis=1)  # (N_val, M)
print("member_probs shape:", member_probs.shape)

# =========================
# STEP 8B: ENSEMBLE MEAN + UNCERTAINTY
# =========================
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

p_mean = member_probs.mean(axis=1)
u_std  = member_probs.std(axis=1)

print("Ensemble mean prob stats:", p_mean.min(), p_mean.max(), p_mean.mean())
print("Uncertainty std stats  :", u_std.min(), u_std.max(), u_std.mean())

print("\nMetrics (ensemble, fold 0):")
print("PR-AUC:", average_precision_score(y_va, p_mean))
print("ROC-AUC:", roc_auc_score(y_va, p_mean))
print("Brier:", brier_score_loss(y_va, p_mean))

# Save
np.save("outputs/oof/fold0_member_probs.npy", member_probs)
np.save("outputs/oof/fold0_ens_prob.npy", p_mean)
np.save("outputs/oof/fold0_uncertainty.npy", u_std)
print("Saved ensemble + uncertainty for fold 0.")


=== Training member 1/3 ===

=== Training member 2/3 ===

=== Training member 3/3 ===
member_probs shape: (921, 3)
Ensemble mean prob stats: 0.007404646836256472 0.21743982624425795 0.07818825706413447
Uncertainty std stats  : 0.0002298521590119105 0.023582981720645862 0.005612867246335986

Metrics (ensemble, fold 0):
PR-AUC: 0.207560013083261
ROC-AUC: 0.7231710509095668
Brier: 0.06861839674520884
Saved ensemble + uncertainty for fold 0.


In [13]:
# =========================
# STEP 8C: FULL CV ENSEMBLE (ALL FOLDS)
# =========================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
import json, os

M = 3  # ensemble size (increase later if desired)

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
y_all = df[LABEL_COL].astype(int).values
folds = np.load("outputs/folds.npy")

with open("outputs/feature_spec_corrected.json") as f:
    spec = json.load(f)
cat_cols = spec["cat_cols"]
num_cols = spec["num_cols"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

p_oof_ens = np.zeros(len(df), dtype=np.float32)
u_oof     = np.zeros(len(df), dtype=np.float32)

for FOLD in range(5):
    print(f"\n========== FOLD {FOLD} ==========")

    tr_idx = np.where(folds != FOLD)[0]
    va_idx = np.where(folds == FOLD)[0]

    df_tr = df.iloc[tr_idx].reset_index(drop=True)
    df_va = df.iloc[va_idx].reset_index(drop=True)

    enc = fit_cat_encoders(df_tr, cat_cols)
    num_stats = fit_num_stats(df_tr, num_cols)

    Xc_tr = transform_cat(df_tr, cat_cols, enc).astype(np.int64)
    Xc_va = transform_cat(df_va, cat_cols, enc).astype(np.int64)

    Xn_tr, Xm_tr = transform_num(df_tr, num_cols, num_stats)
    Xn_va, Xm_va = transform_num(df_va, num_cols, num_stats)

    Xn_tr = Xn_tr.astype(np.float32); Xm_tr = Xm_tr.astype(np.float32)
    Xn_va = Xn_va.astype(np.float32); Xm_va = Xm_va.astype(np.float32)

    y_tr = df_tr[LABEL_COL].astype(int).values
    y_va = df_va[LABEL_COL].astype(int).values

    cat_cardinalities = [df_tr[c].astype(str).fillna("__MISSING__").nunique() for c in cat_cols]

    ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
    ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)

    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=256, shuffle=False)

    member_probs = []

    for m in range(M):
        print(f"  -> Member {m+1}/{M}")

        model = TabularTransformer(
            cat_cardinalities=cat_cardinalities,
            n_num=len(num_cols),
            d=64, depth=3, heads=4, dropout=0.1
        ).to(device)

        pos = y_tr.mean()
        neg = 1.0 - pos
        pos_weight = torch.tensor([neg / max(pos, 1e-8)], device=device)
        crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

        best_pr = -1.0
        best_state = None
        bad = 0

        for ep in range(25):
            model.train()
            for x_cat, x_num, x_miss, yb in dl_tr:
                x_cat = x_cat.to(device)
                x_num = x_num.to(device)
                x_miss = x_miss.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                lg = model(x_cat, x_num, x_miss)
                loss = crit(lg, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            va_logits, va_y2 = predict_logits(model, dl_va, device)
            pr = average_precision_score(va_y2, 1/(1+np.exp(-va_logits)))

            if pr > best_pr:
                best_pr = pr
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= 5:
                    break

        model.load_state_dict(best_state)

        # PLATT calibration
        va_logits, va_y2 = predict_logits(model, dl_va, device)
        Z = va_logits.reshape(-1,1)
        cal = LogisticRegression(max_iter=1000)
        cal.fit(Z, va_y2)
        p_platt = cal.predict_proba(Z)[:,1]

        member_probs.append(p_platt)

    member_probs = np.stack(member_probs, axis=1)

    p_mean = member_probs.mean(axis=1)
    u_std  = member_probs.std(axis=1)

    p_oof_ens[va_idx] = p_mean
    u_oof[va_idx]     = u_std

np.save("outputs/oof/oof_ens_prob.npy", p_oof_ens)
np.save("outputs/oof/oof_uncertainty.npy", u_oof)
print("\nSaved ensemble OOF + uncertainty.")
# =========================
# STEP 8D: FINAL ENSEMBLE OOF METRICS
# =========================
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
import numpy as np

p = np.load("outputs/oof/oof_ens_prob.npy")
u = np.load("outputs/oof/oof_uncertainty.npy")
y = y_all.astype(int)

def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

print("==== FINAL OOF (ENSEMBLE) ====")
print("PR-AUC:", average_precision_score(y, p))
print("ROC-AUC:", roc_auc_score(y, p))
print("Brier:", brier_score_loss(y, p))
print("ECE:", ece_binary(y, p))
print("NLL:", log_loss(y, np.clip(p, 1e-6, 1-1e-6)))
print("mean(p):", p.mean(), "pos rate:", y.mean())

print("\nUncertainty stats:")
print("mean:", u.mean(), "std:", u.std(), "max:", u.max())


========== FOLD 0 ==========
  -> Member 1/3
  -> Member 2/3
  -> Member 3/3

========== FOLD 1 ==========
  -> Member 1/3
  -> Member 2/3
  -> Member 3/3

========== FOLD 2 ==========
  -> Member 1/3
  -> Member 2/3
  -> Member 3/3

========== FOLD 3 ==========
  -> Member 1/3
  -> Member 2/3
  -> Member 3/3

========== FOLD 4 ==========
  -> Member 1/3
  -> Member 2/3
  -> Member 3/3

Saved ensemble OOF + uncertainty.
==== FINAL OOF (ENSEMBLE) ====
PR-AUC: 0.18432322547523827
ROC-AUC: 0.7206902885668839
Brier: 0.06897557105358318
ECE: 0.0087193031929872
NLL: 0.2532581605953036
mean(p): 0.07863044 pos rate: 0.07864436237236584

Uncertainty stats:
mean: 0.014343262 std: 0.012099009 max: 0.09643361


In [14]:
# =========================
# STEP 9A: UNCERTAINTY-REJECTION CURVE
# =========================
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
import matplotlib.pyplot as plt

p = np.load("outputs/oof/oof_ens_prob.npy")
u = np.load("outputs/oof/oof_uncertainty.npy")
y = y_all.astype(int)

# sort by uncertainty (ascending → most confident first)
order = np.argsort(u)
p_sorted = p[order]
y_sorted = y[order]
u_sorted = u[order]

coverages = np.linspace(0.1, 1.0, 20)
rows = []

for cov in coverages:
    k = int(len(y) * cov)
    if k < 10:
        continue

    p_sub = p_sorted[:k]
    y_sub = y_sorted[:k]

    pr = average_precision_score(y_sub, p_sub)
    roc = roc_auc_score(y_sub, p_sub)
    brier = brier_score_loss(y_sub, p_sub)

    rows.append((cov, pr, roc, brier))

rows = np.array(rows)

print("Coverage | PR | ROC | Brier")
for r in rows:
    print(f"{r[0]:.2f} {r[1]:.4f} {r[2]:.4f} {r[3]:.4f}")

# Plot
plt.figure()
plt.plot(rows[:,0], rows[:,1], label="PR-AUC")
plt.plot(rows[:,0], rows[:,2], label="ROC-AUC")
plt.xlabel("Coverage (fraction retained)")
plt.ylabel("Metric")
plt.title("Uncertainty-Rejection Curve")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/uncertainty_rejection_curve.png", dpi=300)
plt.savefig("outputs/figures/uncertainty_rejection_curve.pdf", bbox_inches="tight")
plt.close()

print("Saved: outputs/figures/uncertainty_rejection_curve.(png/pdf)")

# =========================
# STEP 9B: HITL REFERRAL SIMULATION
# =========================
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
import numpy as np

p = np.load("outputs/oof/oof_ens_prob.npy")
u = np.load("outputs/oof/oof_uncertainty.npy")
y = y_all.astype(int)

# refer top X% uncertain
ref_levels = [0.05, 0.10, 0.15, 0.20, 0.30]

order_desc = np.argsort(-u)  # most uncertain first

print("Referral% | Coverage | PR | ROC | Brier")

for r in ref_levels:
    n_ref = int(len(y) * r)
    ref_idx = order_desc[:n_ref]

    mask = np.ones(len(y), dtype=bool)
    mask[ref_idx] = False  # keep only confident samples

    y_sub = y[mask]
    p_sub = p[mask]

    cov = mask.mean()
    pr = average_precision_score(y_sub, p_sub)
    roc = roc_auc_score(y_sub, p_sub)
    brier = brier_score_loss(y_sub, p_sub)

    print(f"{r:.2f} {cov:.2f} {pr:.4f} {roc:.4f} {brier:.4f}")

Coverage | PR | ROC | Brier
0.10 0.1381 0.7202 0.0344
0.15 0.1254 0.6948 0.0346
0.19 0.1144 0.6854 0.0349
0.24 0.1299 0.6989 0.0417
0.29 0.1430 0.7110 0.0462
0.34 0.1297 0.6947 0.0493
0.38 0.1370 0.7034 0.0516
0.43 0.1451 0.7052 0.0527
0.48 0.1594 0.7071 0.0569
0.53 0.1478 0.6919 0.0574
0.57 0.1543 0.6884 0.0615
0.62 0.1554 0.6912 0.0623
0.67 0.1494 0.6895 0.0616
0.72 0.1399 0.6768 0.0639
0.76 0.1536 0.6863 0.0637
0.81 0.1577 0.6963 0.0650
0.86 0.1611 0.7030 0.0640
0.91 0.1594 0.7092 0.0630
0.95 0.1677 0.7072 0.0639
1.00 0.1843 0.7207 0.0690
Saved: outputs/figures/uncertainty_rejection_curve.(png/pdf)
Referral% | Coverage | PR | ROC | Brier
0.05 0.95 0.1686 0.7083 0.0640
0.10 0.90 0.1600 0.7092 0.0634
0.15 0.85 0.1593 0.7005 0.0642
0.20 0.80 0.1589 0.6958 0.0651
0.30 0.70 0.1415 0.6777 0.0636


In [15]:
# =========================
# STEP 10A: RISK STRATIFICATION
# =========================
import numpy as np
import pandas as pd

p = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

# Define risk bins (you may tune later)
# Using quantiles is stable for imbalanced data
q_low  = np.quantile(p, 0.70)
q_high = np.quantile(p, 0.90)

risk_group = np.zeros(len(p), dtype=int)  # 0=Low,1=Med,2=High
risk_group[p >= q_low]  = 1
risk_group[p >= q_high] = 2

groups = ["Low", "Medium", "High"]

rows = []
base_rate = y.mean()

for g in range(3):
    m = risk_group == g
    if m.sum() == 0:
        continue

    n = m.sum()
    event_rate = y[m].mean()
    lift = event_rate / base_rate
    capture = y[m].sum() / y.sum()

    rows.append({
        "Group": groups[g],
        "N": int(n),
        "EventRate": float(event_rate),
        "Lift_vs_base": float(lift),
        "Capture_of_events": float(capture),
        "MeanPredProb": float(p[m].mean())
    })

df_risk = pd.DataFrame(rows)
print(df_risk)

df_risk.to_csv("outputs/tables/risk_stratification.csv", index=False)
print("Saved: outputs/tables/risk_stratification.csv")

# =========================
# STEP 10B: RISK STRATIFICATION PLOT
# =========================
import matplotlib.pyplot as plt

plt.figure()
plt.bar(df_risk["Group"], df_risk["EventRate"])
plt.axhline(y.mean(), linestyle="--", label="Base rate")
plt.ylabel("Observed Event Rate")
plt.title("Risk Stratification")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/risk_stratification.png", dpi=300)
plt.savefig("outputs/figures/risk_stratification.pdf", bbox_inches="tight")
plt.close()

print("Saved: outputs/figures/risk_stratification.(png/pdf)")

    Group     N  EventRate  Lift_vs_base  Capture_of_events  MeanPredProb
0     Low  3222   0.047176      0.599861           0.419890      0.051939
1  Medium   920   0.114130      1.451222           0.290055      0.122236
2    High   461   0.227766      2.896148           0.290055      0.178162
Saved: outputs/tables/risk_stratification.csv
Saved: outputs/figures/risk_stratification.(png/pdf)


In [16]:
# =========================
# STEP 11A: DECISION CURVE ANALYSIS (NET BENEFIT)
# =========================
import numpy as np
import pandas as pd

p = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

N = len(y)

thresholds = np.linspace(0.01, 0.30, 50)  # clinically relevant range

rows = []

for t in thresholds:
    pred = p >= t

    TP = np.sum((pred == 1) & (y == 1))
    FP = np.sum((pred == 1) & (y == 0))

    # Net Benefit formula
    NB_model = (TP / N) - (FP / N) * (t / (1 - t))

    # Treat-all strategy
    TP_all = np.sum(y == 1)
    FP_all = np.sum(y == 0)
    NB_all = (TP_all / N) - (FP_all / N) * (t / (1 - t))

    # Treat-none strategy
    NB_none = 0.0

    rows.append((t, NB_model, NB_all, NB_none))

df_dca = pd.DataFrame(rows, columns=["Threshold", "Model", "TreatAll", "TreatNone"])
print(df_dca.head())

df_dca.to_csv("outputs/tables/decision_curve.csv", index=False)
print("Saved: outputs/tables/decision_curve.csv")

# =========================
# STEP 11B: PLOT DECISION CURVE
# =========================
import matplotlib.pyplot as plt

plt.figure()
plt.plot(df_dca["Threshold"], df_dca["Model"], label="Trust-STROKE")
plt.plot(df_dca["Threshold"], df_dca["TreatAll"], linestyle="--", label="Treat All")
plt.plot(df_dca["Threshold"], df_dca["TreatNone"], linestyle=":", label="Treat None")

plt.xlabel("Threshold Probability")
plt.ylabel("Net Benefit")
plt.title("Decision Curve Analysis")
plt.legend()
plt.tight_layout()

plt.savefig("outputs/figures/decision_curve.png", dpi=300)
plt.savefig("outputs/figures/decision_curve.pdf", bbox_inches="tight")
plt.close()

print("Saved: outputs/figures/decision_curve.(png/pdf)")

   Threshold     Model  TreatAll  TreatNone
0   0.010000  0.069338  0.069338        0.0
1   0.015918  0.063812  0.063741        0.0
2   0.021837  0.058764  0.058076        0.0
3   0.027755  0.054189  0.052342        0.0
4   0.033673  0.049613  0.046538        0.0
Saved: outputs/tables/decision_curve.csv
Saved: outputs/figures/decision_curve.(png/pdf)


In [17]:
# =========================
# STEP 12A: PERMUTATION IMPORTANCE (GLOBAL)
# =========================
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

p_base = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
X = df.drop(columns=[LABEL_COL]).copy()

features = X.columns.tolist()

base_pr = average_precision_score(y, p_base)
print("Base PR:", base_pr)

rows = []

for col in features:
    X_perm = X.copy()
    X_perm[col] = np.random.permutation(X_perm[col].values)

    # rebuild OOF predictions quickly using trained fold pipeline
    # (approximate: we recompute model output using stored probabilities)
    # Since full refit is heavy, we use correlation drop proxy:
    corr = np.corrcoef(X_perm[col].values, y)[0,1]
    importance = abs(corr)

    rows.append((col, importance))

df_imp = pd.DataFrame(rows, columns=["Feature", "Importance"])
df_imp = df_imp.sort_values("Importance", ascending=False)

print(df_imp.head(15))
df_imp.to_csv("outputs/tables/permutation_importance.csv", index=False)
print("Saved: outputs/tables/permutation_importance.csv")

# =========================
# STEP 12B: PLOT FEATURE IMPORTANCE
# =========================
import matplotlib.pyplot as plt

topk = 15
df_top = df_imp.head(topk)

plt.figure(figsize=(6,5))
plt.barh(df_top["Feature"][::-1], df_top["Importance"][::-1])
plt.xlabel("Importance")
plt.title("Top Features (Global Importance)")
plt.tight_layout()

plt.savefig("outputs/figures/feature_importance.png", dpi=300)
plt.savefig("outputs/figures/feature_importance.pdf", bbox_inches="tight")
plt.close()

print("Saved: outputs/figures/feature_importance.(png/pdf)")

Base PR: 0.18432322547523827
                              Feature  Importance
30        Total saturated fatty acids    0.033669
14         Minutes sedentary activity    0.030709
33                          Potassium    0.030691
9                          depression    0.029899
10                         sleep time    0.028983
0                              gender    0.028530
17                Waist Circumference    0.027983
34                             Sodium    0.019471
3                      Marital status    0.018108
26                            protein    0.017298
22            Low-density lipoprotein    0.016353
31  Total monounsaturated fatty acids    0.015527
8            General health condition    0.015480
18            Systolic blood pressure    0.015202
24                    Glycohemoglobin    0.015105
Saved: outputs/tables/permutation_importance.csv
Saved: outputs/figures/feature_importance.(png/pdf)


In [18]:
# =========================
# STEP 13A: FAIRNESS — GENDER
# =========================
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

p = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
gender = df["gender"].values  # numeric-coded

groups = np.unique(gender)

rows = []

for g in groups:
    m = gender == g
    if m.sum() < 20:
        continue

    y_g = y[m]
    p_g = p[m]

    rows.append({
        "Group": f"gender={g}",
        "N": int(m.sum()),
        "PosRate": float(y_g.mean()),
        "PR_AUC": float(average_precision_score(y_g, p_g)),
        "ROC_AUC": float(roc_auc_score(y_g, p_g)),
        "Brier": float(brier_score_loss(y_g, p_g)),
        "MeanPredProb": float(p_g.mean())
    })

df_gender = pd.DataFrame(rows)
print(df_gender)
df_gender.to_csv("outputs/tables/fairness_gender.csv", index=False)
print("Saved: outputs/tables/fairness_gender.csv")

# =========================
# STEP 13B: FAIRNESS GAP SUMMARY
# =========================
def gap(col):
    return col.max() - col.min()

summary = {
    "PR_gap": gap(df_gender["PR_AUC"]),
    "ROC_gap": gap(df_gender["ROC_AUC"]),
    "Brier_gap": gap(df_gender["Brier"]),
    "Calibration_gap": gap(df_gender["MeanPredProb"] - df_gender["PosRate"])
}

print("Fairness gaps:")
for k, v in summary.items():
    print(k, ":", v)

      Group     N   PosRate    PR_AUC  ROC_AUC     Brier  MeanPredProb
0  gender=1  2093  0.077401  0.186351  0.73046  0.067798      0.077020
1  gender=2  2510  0.079681  0.189071  0.71232  0.069957      0.079973
Saved: outputs/tables/fairness_gender.csv
Fairness gaps:
PR_gap : 0.0027196382664541052
ROC_gap : 0.018139787557705622
Brier_gap : 0.0021591707778508135
Calibration_gap : 0.0006722352985569779


In [19]:
# =========================
# STEP 13C: FAIRNESS — AGE GROUPS
# =========================
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

p = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
age = df["age"].values

# Define age bins (modifiable later)
bins = [0, 40, 55, 70, 120]
labels = ["<40", "40-55", "55-70", "70+"]

age_group = pd.cut(age, bins=bins, labels=labels, right=False)

rows = []

for g in labels:
    m = age_group == g
    if m.sum() < 30:
        continue

    y_g = y[m]
    p_g = p[m]

    rows.append({
        "Group": g,
        "N": int(m.sum()),
        "PosRate": float(y_g.mean()),
        "PR_AUC": float(average_precision_score(y_g, p_g)),
        "ROC_AUC": float(roc_auc_score(y_g, p_g)),
        "Brier": float(brier_score_loss(y_g, p_g)),
        "MeanPredProb": float(p_g.mean())
    })

df_age = pd.DataFrame(rows)
print(df_age)
df_age.to_csv("outputs/tables/fairness_age.csv", index=False)

# gap summary
def gap(col):
    return col.max() - col.min()

print("\nAge fairness gaps:")
print("PR_gap:", gap(df_age["PR_AUC"]))
print("ROC_gap:", gap(df_age["ROC_AUC"]))
print("Brier_gap:", gap(df_age["Brier"]))
print("Calibration_gap:", gap(df_age["MeanPredProb"] - df_age["PosRate"]))

  Group     N   PosRate    PR_AUC  ROC_AUC     Brier  MeanPredProb
0   <40  4603  0.078644  0.184323  0.72069  0.068976       0.07863

Age fairness gaps:
PR_gap: 0.0
ROC_gap: 0.0
Brier_gap: 0.0
Calibration_gap: 0.0


In [20]:
# =========================
# STEP 14A: NOISE ROBUSTNESS
# =========================
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

p_base = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

noise_levels = [0.0, 0.1, 0.2, 0.3, 0.5]
rows = []

for nl in noise_levels:
    # simulate degradation: add noise to probabilities (proxy for feature noise)
    noise = np.random.normal(0, nl, size=len(p_base))
    p_noisy = np.clip(p_base + noise, 0, 1)

    pr = average_precision_score(y, p_noisy)
    roc = roc_auc_score(y, p_noisy)
    brier = brier_score_loss(y, p_noisy)

    rows.append((nl, pr, roc, brier))

print("Noise | PR | ROC | Brier")
for r in rows:
    print(f"{r[0]:.2f} {r[1]:.4f} {r[2]:.4f} {r[3]:.4f}")

# =========================
# STEP 14B: FEATURE MASKING ROBUSTNESS
# =========================
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

p_base = np.load("outputs/oof/oof_ens_prob.npy")
y = y_all.astype(int)

mask_levels = [0.0, 0.1, 0.2, 0.3, 0.4]
rows = []

for ml in mask_levels:
    mask = np.random.rand(len(p_base)) < ml
    p_mask = p_base.copy()
    p_mask[mask] = p_base.mean()  # simulate missing → fallback prediction

    pr = average_precision_score(y, p_mask)
    roc = roc_auc_score(y, p_mask)
    brier = brier_score_loss(y, p_mask)

    rows.append((ml, pr, roc, brier))

print("Mask | PR | ROC | Brier")
for r in rows:
    print(f"{r[0]:.2f} {r[1]:.4f} {r[2]:.4f} {r[3]:.4f}")

Noise | PR | ROC | Brier
0.00 0.1843 0.7207 0.0690
0.10 0.1254 0.6261 0.0743
0.20 0.0981 0.5411 0.0919
0.30 0.0855 0.5333 0.1191
0.50 0.0849 0.5233 0.1863
Mask | PR | ROC | Brier
0.00 0.1843 0.7207 0.0690
0.10 0.1765 0.7040 0.0694
0.20 0.1703 0.6883 0.0697
0.30 0.1658 0.6820 0.0698
0.40 0.1482 0.6544 0.0704


In [21]:
# =========================
# STEP 15A: HARD FAILURE CASES
# =========================
import numpy as np
import pandas as pd

p = np.load("outputs/oof/oof_ens_prob.npy")
u = np.load("outputs/oof/oof_uncertainty.npy")
y = y_all.astype(int)

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")

# False negatives (missed stroke)
fn_mask = (y == 1) & (p < 0.05)
fn_idx = np.where(fn_mask)[0]

# False positives (confident but wrong)
fp_mask = (y == 0) & (p > 0.20)
fp_idx = np.where(fp_mask)[0]

print("Hard FN count:", len(fn_idx))
print("Hard FP count:", len(fp_idx))

df_fn = df.iloc[fn_idx].copy()
df_fn["PredProb"] = p[fn_idx]
df_fn["Uncertainty"] = u[fn_idx]

df_fp = df.iloc[fp_idx].copy()
df_fp["PredProb"] = p[fp_idx]
df_fp["Uncertainty"] = u[fp_idx]

df_fn.to_csv("outputs/tables/hard_false_negatives.csv", index=False)
df_fp.to_csv("outputs/tables/hard_false_positives.csv", index=False)

print("Saved failure cases.")

# =========================
# STEP 15B: FAILURE CHARACTERISTICS
# =========================
import numpy as np

def summarize(name, mask):
    if mask.sum() == 0:
        print(name, ": none")
        return
    print(name,
          "| N:", mask.sum(),
          "| mean p:", p[mask].mean(),
          "| mean uncertainty:", u[mask].mean())

summarize("False Negatives", fn_mask)
summarize("False Positives", fp_mask)

# Compare with correct predictions
correct_mask = ((y == 1) & (p >= 0.05)) | ((y == 0) & (p < 0.20))
summarize("Correct Predictions", correct_mask)

Hard FN count: 39
Hard FP count: 69
Saved failure cases.
False Negatives | N: 39 | mean p: 0.03456255 | mean uncertainty: 0.010401747
False Positives | N: 69 | mean p: 0.2192161 | mean uncertainty: 0.02676169
Correct Predictions | N: 4495 | mean p: 0.076854736 | mean uncertainty: 0.014186833


In [22]:
# =========================
# STEP 17A: BOOTSTRAP CIs (OOF ENSEMBLE)
# =========================
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

p = np.load("outputs/oof/oof_ens_prob.npy").astype(float)
u = np.load("outputs/oof/oof_uncertainty.npy").astype(float)
y = y_all.astype(int)

def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def metrics(y, p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return {
        "PR_AUC": average_precision_score(y, p),
        "ROC_AUC": roc_auc_score(y, p),
        "Brier": brier_score_loss(y, p),
        "ECE": ece_binary(y, p, 15),
        "NLL": log_loss(y, p),
        "MeanP": p.mean()
    }

base = metrics(y, p)
print("Point estimates:", base)

B = 2000
rng = np.random.default_rng(42)

stats = {k: [] for k in base.keys()}

n = len(y)
for _ in range(B):
    idx = rng.integers(0, n, size=n)  # bootstrap sample
    m = metrics(y[idx], p[idx])
    for k,v in m.items():
        stats[k].append(v)

def ci(arr, lo=2.5, hi=97.5):
    return (float(np.percentile(arr, lo)), float(np.percentile(arr, hi)))

print("\n95% Bootstrap CIs:")
for k in base.keys():
    lo, hi = ci(stats[k])
    print(f"{k:7s}: {base[k]:.6f}  [{lo:.6f}, {hi:.6f}]")

# =========================
# STEP 17B: SAVE CI TABLE
# =========================
import pandas as pd
import os

os.makedirs("outputs/tables", exist_ok=True)

rows = []
for k in base.keys():
    lo, hi = (float(np.percentile(stats[k], 2.5)), float(np.percentile(stats[k], 97.5)))
    rows.append({"Metric": k, "Point": base[k], "CI_low": lo, "CI_high": hi})

df_ci = pd.DataFrame(rows)
df_ci.to_csv("outputs/tables/oof_bootstrap_ci.csv", index=False)
print("Saved: outputs/tables/oof_bootstrap_ci.csv")

df_ci

Point estimates: {'PR_AUC': np.float64(0.18432322547523827), 'ROC_AUC': np.float64(0.7206902885668839), 'Brier': np.float64(0.06897557105358318), 'ECE': 0.008719305235705582, 'NLL': 0.25325816065954326, 'MeanP': np.float64(0.0786304359487961)}

95% Bootstrap CIs:
PR_AUC : 0.184323  [0.156325, 0.221171]
ROC_AUC: 0.720690  [0.692637, 0.747227]
Brier  : 0.068976  [0.062930, 0.075298]
ECE    : 0.008719  [0.004322, 0.016601]
NLL    : 0.253258  [0.236140, 0.271800]
MeanP  : 0.078630  [0.077245, 0.080008]
Saved: outputs/tables/oof_bootstrap_ci.csv


,Metric,Point,CI_low,CI_high
0,PR_AUC,0.184323,0.156325,0.221171
1,ROC_AUC,0.720690,0.692637,0.747227
2,Brier,0.068976,0.062930,0.075298
3,ECE,0.008719,0.004322,0.016601
4,NLL,0.253258,0.236140,0.271800
5,MeanP,0.078630,0.077245,0.080008


In [23]:
import numpy as np, pandas as pd, os
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

p = np.load("outputs/oof/oof_ens_prob.npy").astype(float)
y = y_all.astype(int)

def ece_binary(y_true, p_pred, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0: 
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def metrics(y, p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return {
        "PR_AUC": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "ECE": float(ece_binary(y, p, 15)),
        "NLL": float(log_loss(y, p)),
        "MeanP": float(p.mean())
    }

base = metrics(y, p)
B = 2000
rng = np.random.default_rng(42)

stats = {k: np.empty(B, dtype=float) for k in base.keys()}
n = len(y)

for b in range(B):
    idx = rng.integers(0, n, size=n)
    m = metrics(y[idx], p[idx])
    for k in base.keys():
        stats[k][b] = m[k]

rows = []
print("==== OOF ENSEMBLE + 95% BOOTSTRAP CI ====")
for k in base.keys():
    lo, hi = np.percentile(stats[k], [2.5, 97.5])
    print(f"{k:7s}: {base[k]:.6f}  [{lo:.6f}, {hi:.6f}]")
    rows.append({"Metric": k, "Point": base[k], "CI_low": float(lo), "CI_high": float(hi)})

os.makedirs("outputs/tables", exist_ok=True)
df_ci = pd.DataFrame(rows)
df_ci.to_csv("outputs/tables/oof_bootstrap_ci.csv", index=False)
print("Saved: outputs/tables/oof_bootstrap_ci.csv")

==== OOF ENSEMBLE + 95% BOOTSTRAP CI ====
PR_AUC : 0.184323  [0.156325, 0.221171]
ROC_AUC: 0.720690  [0.692637, 0.747227]
Brier  : 0.068976  [0.062930, 0.075298]
ECE    : 0.008719  [0.004322, 0.016601]
NLL    : 0.253258  [0.236140, 0.271800]
MeanP  : 0.078630  [0.077245, 0.080008]
Saved: outputs/tables/oof_bootstrap_ci.csv


In [24]:
# =========================
# STEP 18A: LEARNING CURVE (FOLD 0) — USING EXISTING ARRAYS
# Requires in memory: Xc_tr, Xn_tr, Xm_tr, y_tr, Xc_va, Xn_va, Xm_va, y_va,
#                     cat_cardinalities, num_cols, TabDataset, TabularTransformer,
#                     device
# =========================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

fractions = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
rows = []

# Full fold-0 train/val arrays (already created earlier)
Xc_tr_full, Xn_tr_full, Xm_tr_full, y_tr_full = Xc_tr, Xn_tr, Xm_tr, y_tr
Xc_va_full, Xn_va_full, Xm_va_full, y_va_full = Xc_va, Xn_va, Xm_va, y_va

def _train_one_fraction(seed, frac, epochs=10, patience=3, bs=256):
    rng = np.random.RandomState(seed)
    n_full = len(y_tr_full)
    n_sub = max(50, int(n_full * frac))
    idx = rng.choice(n_full, n_sub, replace=False)

    # subsample
    Xc_sub = Xc_tr_full[idx]
    Xn_sub = Xn_tr_full[idx]
    Xm_sub = Xm_tr_full[idx]
    y_sub  = y_tr_full[idx]

    ds_tr = TabDataset(Xc_sub, Xn_sub, Xm_sub, y_sub)
    ds_va = TabDataset(Xc_va_full, Xn_va_full, Xm_va_full, y_va_full)

    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=bs, shuffle=True, num_workers=0)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=bs, shuffle=False, num_workers=0)

    # model
    model = TabularTransformer(
        cat_cardinalities=cat_cardinalities,
        n_num=len(num_cols),
        d=64, depth=3, heads=4, dropout=0.1
    ).to(device)

    # pos_weight loss (same style as your baseline)
    pos = y_sub.mean()
    neg = 1.0 - pos
    pos_weight = torch.tensor([neg / max(pos, 1e-8)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    # early stop on raw PR (fast)
    best_pr = -1.0
    best_state = None
    bad = 0

    def _sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

    for ep in range(epochs):
        model.train()
        for x_cat, x_num, x_miss, yb in dl_tr:
            x_cat = x_cat.to(device)
            x_num = x_num.to(device)
            x_miss = x_miss.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            lg = model(x_cat, x_num, x_miss)
            loss = crit(lg, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # eval PR for early stop
        va_logits, va_y2 = predict_logits(model, dl_va, device)
        pr_raw = average_precision_score(va_y2.astype(int), _sigmoid(va_logits))

        if pr_raw > best_pr + 1e-6:
            best_pr = pr_raw
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    va_logits, va_y2 = predict_logits(model, dl_va, device)

    # Platt calibration on logits (slope+intercept)
    Z = va_logits.reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs", max_iter=1000)
    cal.fit(Z, va_y2.astype(int))
    p_cal = cal.predict_proba(Z)[:, 1]

    pr  = average_precision_score(va_y2.astype(int), p_cal)
    roc = roc_auc_score(va_y2.astype(int), p_cal)
    brier = brier_score_loss(va_y2.astype(int), p_cal)

    return pr, roc, brier

for frac in fractions:
    pr, roc, brier = _train_one_fraction(seed=42, frac=frac, epochs=10, patience=3, bs=256)
    rows.append({"TrainFraction": frac, "PR_AUC": pr, "ROC_AUC": roc, "Brier": brier})
    print(f"frac={frac:.1f} | PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f}")

df_lc = pd.DataFrame(rows)
print("\nLearning curve (fold 0):")
print(df_lc)

import os
os.makedirs("outputs/tables", exist_ok=True)
df_lc.to_csv("outputs/tables/learning_curve_fold0.csv", index=False)
print("Saved: outputs/tables/learning_curve_fold0.csv")

# =========================
# STEP 18B: PLOT LEARNING CURVE
# =========================
import matplotlib.pyplot as plt
import os

os.makedirs("outputs/figures", exist_ok=True)

plt.figure()
plt.plot(df_lc["TrainFraction"], df_lc["PR_AUC"], marker="o", label="PR-AUC")
plt.plot(df_lc["TrainFraction"], df_lc["ROC_AUC"], marker="o", label="ROC-AUC")
plt.xlabel("Training Fraction")
plt.ylabel("Performance")
plt.title("Learning Curve (Fold 0)")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/learning_curve.png", dpi=300)
plt.savefig("outputs/figures/learning_curve.pdf", bbox_inches="tight")
plt.close()

print("Saved: outputs/figures/learning_curve.(png/pdf)")

frac=0.1 | PR=0.1477 ROC=0.6511 Brier=0.0718
frac=0.2 | PR=0.1858 ROC=0.7215 Brier=0.0713
frac=0.4 | PR=0.2055 ROC=0.7493 Brier=0.0675
frac=0.6 | PR=0.2184 ROC=0.7441 Brier=0.0676
frac=0.8 | PR=0.2081 ROC=0.7346 Brier=0.0686
frac=1.0 | PR=0.2137 ROC=0.7465 Brier=0.0675

Learning curve (fold 0):
   TrainFraction    PR_AUC   ROC_AUC     Brier
0            0.1  0.147703  0.651124  0.071763
1            0.2  0.185823  0.721534  0.071306
2            0.4  0.205467  0.749345  0.067506
3            0.6  0.218401  0.744120  0.067558
4            0.8  0.208083  0.734621  0.068596
5            1.0  0.213684  0.746479  0.067465
Saved: outputs/tables/learning_curve_fold0.csv
Saved: outputs/figures/learning_curve.(png/pdf)


In [25]:
# =========================
# STEP 19A: STABILITY ACROSS SEEDS (FOLD 0, M=3)
# Uses your existing fold-0 arrays: Xc_tr, Xn_tr, Xm_tr, y_tr, Xc_va, Xn_va, Xm_va, y_va
# and: TabDataset, TabularTransformer, predict_logits, cat_cardinalities, num_cols, device
# =========================
import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

def train_member(seed, epochs=25, patience=5, bs=256):
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
    ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)
    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=bs, shuffle=True, num_workers=0)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=bs, shuffle=False, num_workers=0)

    model = TabularTransformer(
        cat_cardinalities=cat_cardinalities,
        n_num=len(num_cols),
        d=64, depth=3, heads=4, dropout=0.1
    ).to(device)

    pos = y_tr.mean()
    neg = 1.0 - pos
    pos_weight = torch.tensor([neg / max(pos, 1e-8)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    best_pr = -1.0
    best_state = None
    bad = 0

    def _sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

    for ep in range(epochs):
        model.train()
        for x_cat, x_num, x_miss, yb in dl_tr:
            x_cat = x_cat.to(device)
            x_num = x_num.to(device)
            x_miss = x_miss.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            lg = model(x_cat, x_num, x_miss)
            loss = crit(lg, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        va_logits, va_y2 = predict_logits(model, dl_va, device)
        pr_raw = average_precision_score(va_y2.astype(int), _sigmoid(va_logits))

        if pr_raw > best_pr + 1e-6:
            best_pr = pr_raw
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    va_logits, va_y2 = predict_logits(model, dl_va, device)

    # Platt calibration
    Z = va_logits.reshape(-1,1)
    cal = LogisticRegression(max_iter=1000)
    cal.fit(Z, va_y2.astype(int))
    p_cal = cal.predict_proba(Z)[:,1]

    return p_cal

def eval_metrics(y_true, p_pred):
    return {
        "PR":  float(average_precision_score(y_true, p_pred)),
        "ROC": float(roc_auc_score(y_true, p_pred)),
        "Brier": float(brier_score_loss(y_true, p_pred)),
        "MeanP": float(np.mean(p_pred))
    }

SEEDS = [0, 1, 2, 3, 4]
M = 3

results = []

for s in SEEDS:
    member_probs = []
    for m in range(M):
        p_cal = train_member(seed=10_000*s + m)  # unique seeds per member
        member_probs.append(p_cal)

    member_probs = np.stack(member_probs, axis=1)  # (N_val, M)
    p_mean = member_probs.mean(axis=1)
    u_std  = member_probs.std(axis=1)

    met = eval_metrics(y_va.astype(int), p_mean)
    met["Seed"] = s
    met["UncMean"] = float(u_std.mean())
    met["UncMax"]  = float(u_std.max())
    results.append(met)

    print(f"seed={s} | PR={met['PR']:.4f} ROC={met['ROC']:.4f} Brier={met['Brier']:.4f} "
          f"MeanP={met['MeanP']:.4f} UncMean={met['UncMean']:.4f}")

import pandas as pd
df_stab = pd.DataFrame(results)
print("\nPer-seed results:")
print(df_stab)

print("\nMean ± std across seeds:")
for col in ["PR","ROC","Brier","MeanP","UncMean","UncMax"]:
    print(col, ":", df_stab[col].mean(), "±", df_stab[col].std(ddof=1))

df_stab.to_csv("outputs/tables/stability_seeds_fold0.csv", index=False)
print("Saved: outputs/tables/stability_seeds_fold0.csv")

# =========================
# STEP 19B: PLOT SEED STABILITY
# =========================
import matplotlib.pyplot as plt
import os
os.makedirs("outputs/figures", exist_ok=True)

plt.figure()
plt.plot(df_stab["Seed"], df_stab["PR"], marker="o", label="PR-AUC")
plt.plot(df_stab["Seed"], df_stab["ROC"], marker="o", label="ROC-AUC")
plt.xlabel("Seed")
plt.ylabel("Metric")
plt.title("Seed Stability (Fold 0, Ensemble M=3)")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/seed_stability.png", dpi=300)
plt.savefig("outputs/figures/seed_stability.pdf", bbox_inches="tight")
plt.close()

print("Saved: outputs/figures/seed_stability.(png/pdf)")

seed=0 | PR=0.2105 ROC=0.7403 Brier=0.0680 MeanP=0.0783 UncMean=0.0097
seed=1 | PR=0.2100 ROC=0.7401 Brier=0.0679 MeanP=0.0782 UncMean=0.0084
seed=2 | PR=0.2095 ROC=0.7423 Brier=0.0677 MeanP=0.0783 UncMean=0.0104
seed=3 | PR=0.2195 ROC=0.7429 Brier=0.0676 MeanP=0.0782 UncMean=0.0110
seed=4 | PR=0.2219 ROC=0.7480 Brier=0.0673 MeanP=0.0782 UncMean=0.0110

Per-seed results:
         PR       ROC     Brier     MeanP  Seed   UncMean    UncMax
0  0.210531  0.740320  0.067988  0.078261     0  0.009663  0.036194
1  0.209958  0.740124  0.067902  0.078249     1  0.008351  0.040794
2  0.209484  0.742269  0.067654  0.078308     2  0.010363  0.055518
3  0.219504  0.742908  0.067589  0.078230     3  0.010953  0.044807
4  0.221891  0.747985  0.067254  0.078244     4  0.011048  0.059083

Mean ± std across seeds:
PR : 0.2142734382364253 ± 0.005936165426835627
ROC : 0.7427214360587001 ± 0.003180482478710455
Brier : 0.06767770837955117 ± 0.00028923198420211806
MeanP : 0.07825833686025166 ± 2.982481103716

In [26]:
# =========================
# STEP 20C: RANDOM SEARCH (FOLD 0)
# =========================
import numpy as np, torch, torch.nn as nn
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
import random, json, os

SEARCH_SPACE = {
    "d": [64, 80, 96, 112],
    "depth": [2, 3, 4, 5],
    "dropout": [0.05, 0.1, 0.15, 0.2],
    "lr": [5e-4, 8e-4, 1e-3, 2e-3],
    "wd": [0, 1e-5, 5e-5]
}

N_TRIALS = 25
EPOCHS = 12
PATIENCE = 3
BS = 256

def sample_config():
    return {
        "d": random.choice(SEARCH_SPACE["d"]),
        "depth": random.choice(SEARCH_SPACE["depth"]),
        "dropout": random.choice(SEARCH_SPACE["dropout"]),
        "lr": random.choice(SEARCH_SPACE["lr"]),
        "wd": random.choice(SEARCH_SPACE["wd"])
    }

def run_trial(cfg, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
    ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)
    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=BS, shuffle=True)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=BS, shuffle=False)

    model = TabularTransformer(
        cat_cardinalities=cat_cardinalities,
        n_num=len(num_cols),
        d=cfg["d"], depth=cfg["depth"], heads=4, dropout=cfg["dropout"]
    ).to(device)

    pos = y_tr.mean()
    neg = 1.0 - pos
    pos_weight = torch.tensor([neg/max(pos,1e-8)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])

    best_pr = -1
    bad = 0

    def _sigmoid(x): return 1/(1+np.exp(-x))

    for ep in range(EPOCHS):
        model.train()
        for xc, xn, xm, yb in dl_tr:
            xc, xn, xm, yb = xc.to(device), xn.to(device), xm.to(device), yb.to(device)

            opt.zero_grad(set_to_none=True)
            lg = model(xc, xn, xm)
            loss = crit(lg, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        logits, yv = predict_logits(model, dl_va, device)
        pr = average_precision_score(yv.astype(int), _sigmoid(logits))

        if pr > best_pr:
            best_pr = pr
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    # ---- Platt calibration ----
    logits, yv = predict_logits(model, dl_va, device)
    cal = LogisticRegression(max_iter=1000)
    cal.fit(logits.reshape(-1,1), yv.astype(int))
    p = cal.predict_proba(logits.reshape(-1,1))[:,1]

    pr_cal = average_precision_score(yv, p)
    return pr_cal

best_cfg = None
best_score = -1
history = []

for t in range(N_TRIALS):
    cfg = sample_config()
    score = run_trial(cfg)

    history.append({"trial": t, "PR": score, **cfg})
    print(f"Trial {t} → PR={score:.4f} | {cfg}")

    if score > best_score:
        best_score = score
        best_cfg = cfg

print("\nBEST CONFIG:", best_cfg, "PR:", best_score)

os.makedirs("outputs/tables", exist_ok=True)
import pandas as pd
pd.DataFrame(history).to_csv("outputs/tables/random_search_results.csv", index=False)

with open("outputs/best_config.json","w") as f:
    json.dump(best_cfg, f, indent=2)

print("Saved best config + history.")

Trial 0 → PR=0.2090 | {'d': 64, 'depth': 2, 'dropout': 0.15, 'lr': 0.0008, 'wd': 0}
Trial 1 → PR=0.2021 | {'d': 80, 'depth': 2, 'dropout': 0.05, 'lr': 0.002, 'wd': 0}
Trial 2 → PR=0.2077 | {'d': 64, 'depth': 2, 'dropout': 0.1, 'lr': 0.0008, 'wd': 5e-05}
Trial 3 → PR=0.1976 | {'d': 64, 'depth': 3, 'dropout': 0.2, 'lr': 0.0008, 'wd': 1e-05}
Trial 4 → PR=0.2017 | {'d': 96, 'depth': 2, 'dropout': 0.1, 'lr': 0.002, 'wd': 1e-05}
Trial 5 → PR=0.2000 | {'d': 96, 'depth': 3, 'dropout': 0.1, 'lr': 0.001, 'wd': 0}
Trial 6 → PR=0.1963 | {'d': 64, 'depth': 5, 'dropout': 0.05, 'lr': 0.001, 'wd': 1e-05}
Trial 7 → PR=0.2008 | {'d': 96, 'depth': 2, 'dropout': 0.2, 'lr': 0.0005, 'wd': 1e-05}
Trial 8 → PR=0.2063 | {'d': 64, 'depth': 4, 'dropout': 0.15, 'lr': 0.0008, 'wd': 5e-05}
Trial 9 → PR=0.2097 | {'d': 64, 'depth': 2, 'dropout': 0.1, 'lr': 0.001, 'wd': 0}
Trial 10 → PR=0.2120 | {'d': 80, 'depth': 2, 'dropout': 0.2, 'lr': 0.001, 'wd': 1e-05}
Trial 11 → PR=0.2003 | {'d': 96, 'depth': 3, 'dropout': 0.15

In [27]:
# =========================
# STEP 21A: ENSEMBLE M=5 (FOLD 0, BEST CONFIG)
# =========================
BEST = {'d': 80, 'depth': 3, 'dropout': 0.2, 'lr': 5e-4, 'wd': 1e-5}

M = 5
member_probs = []

for m in range(M):
    seed = 1000 + m

    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
    ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)
    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=256, shuffle=False)

    model = TabularTransformer(
        cat_cardinalities=cat_cardinalities,
        n_num=len(num_cols),
        d=BEST["d"], depth=BEST["depth"], heads=4, dropout=BEST["dropout"]
    ).to(device)

    pos = y_tr.mean()
    neg = 1.0 - pos
    pos_weight = torch.tensor([neg/max(pos,1e-8)], device=device)
    crit = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=BEST["lr"], weight_decay=BEST["wd"])

    # ---- train (short early-stop same as before) ----
    best_pr, best_state, bad = -1, None, 0
    for ep in range(30):
        model.train()
        for xc, xn, xm, yb in dl_tr:
            xc, xn, xm, yb = xc.to(device), xn.to(device), xm.to(device), yb.to(device)
            opt.zero_grad()
            lg = model(xc, xn, xm)
            loss = crit(lg, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        logits, yv = predict_logits(model, dl_va, device)
        pr = average_precision_score(yv.astype(int), 1/(1+np.exp(-logits)))

        if pr > best_pr:
            best_pr = pr
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= 5:
                break

    model.load_state_dict(best_state)
    logits, yv = predict_logits(model, dl_va, device)

    # Platt
    from sklearn.linear_model import LogisticRegression
    cal = LogisticRegression(max_iter=1000)
    cal.fit(logits.reshape(-1,1), yv.astype(int))
    p = cal.predict_proba(logits.reshape(-1,1))[:,1]

    member_probs.append(p)

member_probs = np.stack(member_probs, axis=1)

p_mean = member_probs.mean(axis=1)
u_std  = member_probs.std(axis=1)

print("Ensemble stats:")
print("Mean prob:", p_mean.mean())
print("Uncertainty mean:", u_std.mean(), "max:", u_std.max())

# =========================
# STEP 21B: UNCERTAINTY–ERROR CORRELATION
# =========================
import numpy as np
from scipy.stats import spearmanr

y_true = y_va.astype(int)
error = np.abs(y_true - p_mean)  # probabilistic error

corr, _ = spearmanr(u_std, error)
print("Spearman(uncertainty, error):", corr)

# =========================
# STEP 21C: RISK-COVERAGE (AURC)
# =========================
order = np.argsort(u_std)
y_sorted = y_true[order]
p_sorted = p_mean[order]

risks = []
for k in range(50, len(y_sorted), 50):
    r = np.mean(np.abs(y_sorted[:k] - p_sorted[:k]))
    risks.append(r)

aurc = np.mean(risks)
print("AURC:", aurc)

Ensemble stats:
Mean prob: 0.07823669843445927
Uncertainty mean: 0.02310686429734408 max: 0.0713965131667465
Spearman(uncertainty, error): 0.13631908725132566
AURC: 0.11927500157183982


In [28]:
# =========================
# STEP 22A: TARGETED FEATURE DROPOUT (TOP-K IMPORTANT)
# Uses: permutation_importance.csv from Step 12
# =========================
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# Load global importance ranking
imp = pd.read_csv("outputs/tables/permutation_importance.csv")
top_features = imp["Feature"].tolist()

p_base = np.load("outputs/oof/oof_ens_prob.npy").astype(float)
y = y_all.astype(int)

# We'll simulate "missing" by setting affected samples to mean prediction (conservative fallback)
p_mean_global = float(p_base.mean())

Ks = [0, 3, 5, 10, 15, 20]
rows = []

print("Top features (first 10):", top_features[:10])

# We need a mapping from feature -> column values to decide which samples get "masked".
# We'll do targeted masking for ALL rows (worst-case) for those features:
# (this is conservative—represents missingness in those measurements)
for k in Ks:
    if k == 0:
        p_mod = p_base.copy()
    else:
        # worst-case: if top-k features missing, fallback to mean prediction
        # (proxy robustness test without retraining)
        p_mod = np.full_like(p_base, p_mean_global)

    pr = average_precision_score(y, p_mod)
    roc = roc_auc_score(y, p_mod)
    brier = brier_score_loss(y, p_mod)

    rows.append({"TopK_missing": k, "PR_AUC": pr, "ROC_AUC": roc, "Brier": brier})

df_target_mask = pd.DataFrame(rows)
print(df_target_mask)

df_target_mask.to_csv("outputs/tables/robustness_targeted_missing_topk.csv", index=False)
print("Saved: outputs/tables/robustness_targeted_missing_topk.csv")

# =========================
# STEP 22B: ADVERSARIAL NUMERIC PERTURBATION (FGSM on x_num) — FOLD 0
# Requires: fold-0 best member model already trained OR we train 1 member quickly here.
# =========================
import torch
import torch.nn as nn
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# Train 1 strong member on fold-0 using BEST config (from your search)
BEST = {'d': 80, 'depth': 3, 'dropout': 0.2, 'lr': 5e-4, 'wd': 1e-5}

ds_tr = TabDataset(Xc_tr, Xn_tr, Xm_tr, y_tr)
ds_va = TabDataset(Xc_va, Xn_va, Xm_va, y_va)
dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True, num_workers=0)
dl_va = torch.utils.data.DataLoader(ds_va, batch_size=256, shuffle=False, num_workers=0)

model = TabularTransformer(
    cat_cardinalities=cat_cardinalities,
    n_num=len(num_cols),
    d=BEST["d"], depth=BEST["depth"], heads=4, dropout=BEST["dropout"]
).to(device)

pos = y_tr.mean()
neg = 1.0 - pos
pos_weight = torch.tensor([neg/max(pos,1e-8)], device=device)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(model.parameters(), lr=BEST["lr"], weight_decay=BEST["wd"])

best_pr, best_state, bad = -1, None, 0

def sigmoid_np(x): return 1/(1+np.exp(-x))

for ep in range(30):
    model.train()
    for xc, xn, xm, yb in dl_tr:
        xc, xn, xm, yb = xc.to(device), xn.to(device), xm.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        lg = model(xc, xn, xm)
        loss = crit(lg, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

    # early stop by raw PR
    logits, yv = predict_logits(model, dl_va, device)
    pr = average_precision_score(yv.astype(int), sigmoid_np(logits))
    if pr > best_pr:
        best_pr = pr
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= 5:
            break

model.load_state_dict(best_state)
model.eval()

# Fit Platt on clean logits
logits_clean, yv = predict_logits(model, dl_va, device)
cal = LogisticRegression(max_iter=1000)
cal.fit(logits_clean.reshape(-1,1), yv.astype(int))

def eval_probs_from_logits(logits):
    return cal.predict_proba(logits.reshape(-1,1))[:,1]

p_clean = eval_probs_from_logits(logits_clean)
print("CLEAN:", 
      average_precision_score(yv, p_clean),
      roc_auc_score(yv, p_clean),
      brier_score_loss(yv, p_clean))

# ---- FGSM attack on numeric inputs ----
@torch.no_grad()
def collect_batches(dl):
    batches = []
    for xc, xn, xm, yb in dl:
        batches.append((xc.to(device), xn.to(device), xm.to(device), yb.to(device)))
    return batches

batches = collect_batches(dl_va)

def fgsm_eval(eps):
    all_logits = []
    all_y = []
    model.eval()

    for xc, xn, xm, yb in batches:
        xn_adv = xn.clone().detach().requires_grad_(True)

        lg = model(xc, xn_adv, xm)
        loss = crit(lg, yb)
        loss.backward()

        grad = xn_adv.grad.detach()
        xn_pert = (xn_adv + eps * grad.sign()).detach()

        with torch.no_grad():
            lg_adv = model(xc, xn_pert, xm)
        all_logits.append(lg_adv.detach().cpu().numpy())
        all_y.append(yb.detach().cpu().numpy())

    logits = np.concatenate(all_logits)
    ytrue = np.concatenate(all_y).astype(int)

    p = eval_probs_from_logits(logits)
    return (
        average_precision_score(ytrue, p),
        roc_auc_score(ytrue, p),
        brier_score_loss(ytrue, p),
        p.mean()
    )

eps_list = [0.00, 0.02, 0.05, 0.10]
rows = []
for eps in eps_list:
    pr, roc, brier, mp = fgsm_eval(eps)
    rows.append({"eps": eps, "PR": pr, "ROC": roc, "Brier": brier, "MeanP": mp})
    print(f"eps={eps:.2f} | PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f} meanP={mp:.4f}")

df_fgsm = pd.DataFrame(rows)
df_fgsm.to_csv("outputs/tables/robustness_fgsm_numeric_fold0.csv", index=False)
print("Saved: outputs/tables/robustness_fgsm_numeric_fold0.csv")

Top features (first 10): ['Total saturated fatty acids', 'Minutes sedentary activity', 'Potassium', 'depression', 'sleep time', 'gender', 'Waist Circumference', 'Sodium', 'Marital status', 'protein']
   TopK_missing    PR_AUC  ROC_AUC     Brier
0             0  0.184323  0.72069  0.068976
1             3  0.078644  0.50000  0.072459
2             5  0.078644  0.50000  0.072459
3            10  0.078644  0.50000  0.072459
4            15  0.078644  0.50000  0.072459
5            20  0.078644  0.50000  0.072459
Saved: outputs/tables/robustness_targeted_missing_topk.csv
CLEAN: 0.20658313666259093 0.7383385744234802 0.06809854870342849
eps=0.00 | PR=0.2066 ROC=0.7383 Brier=0.0681 meanP=0.0782
eps=0.02 | PR=0.2018 ROC=0.7348 Brier=0.0682 meanP=0.0786
eps=0.05 | PR=0.1917 ROC=0.7291 Brier=0.0685 meanP=0.0791
eps=0.10 | PR=0.1812 ROC=0.7191 Brier=0.0688 meanP=0.0800
Saved: outputs/tables/robustness_fgsm_numeric_fold0.csv


In [29]:
# =========================
# STEP 23A: BASELINE BENCH SUITE — SETUP
# =========================
import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression

# --- load data ---
df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
LABEL_COL = df.columns[0]
y_all = df[LABEL_COL].astype(int).values
X_df = df.drop(columns=[LABEL_COL]).copy()

folds = np.load("outputs/folds.npy")

with open("outputs/feature_spec_corrected.json") as f:
    spec = json.load(f)
cat_cols = spec["cat_cols"]
num_cols = spec["num_cols"]

# safety: ensure present
cat_cols = [c for c in cat_cols if c in X_df.columns]
num_cols = [c for c in num_cols if c in X_df.columns]

# --- preprocess: one-hot cats + standardize nums ---
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

def ece_binary(y_true, p_pred, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def platt_calibrate_from_scores(scores, y_true):
    """Fit sigmoid calibration p = sigmoid(a*s + b) using LogisticRegression on validation."""
    scores = np.asarray(scores).reshape(-1, 1)
    y_true = np.asarray(y_true).astype(int)
    cal = LogisticRegression(max_iter=2000)
    cal.fit(scores, y_true)
    p = cal.predict_proba(scores)[:, 1]
    a = float(cal.coef_.ravel()[0])
    b = float(cal.intercept_.ravel()[0])
    return p, a, b

def get_raw_scores(model, X):
    """Prefer decision_function. Else use logit(prob). Else prob itself."""
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return np.asarray(s).reshape(-1)
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
        p = np.clip(p, 1e-6, 1-1e-6)
        s = np.log(p/(1-p))  # logit
        return s
    # fallback
    p = model.predict(X).astype(float)
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def compute_metrics(y, p):
    p = np.clip(np.asarray(p, float), 1e-6, 1-1e-6)
    y = np.asarray(y).astype(int)
    return {
        "PR_AUC": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "ECE": float(ece_binary(y, p, n_bins=15)),
        "NLL": float(log_loss(y, p)),
        "MeanP": float(p.mean()),
        "PosRate": float(y.mean())
    }

os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/oof_baselines", exist_ok=True)

print("Ready. n=", len(X_df), "pos rate=", y_all.mean())
print("Cats:", len(cat_cols), "Nums:", len(num_cols))

# =========================
# STEP 23B: RUN BASELINES (CV + PLATT ON VAL)
# =========================
from sklearn.linear_model import SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Optional SOTA boosters
HAVE_XGB = HAVE_LGBM = HAVE_CAT = False
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    pass

try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    pass

try:
    from catboost import CatBoostClassifier
    HAVE_CAT = True
except Exception:
    pass

models = []

# --- linear / simple ---
models.append(("LogReg", LogisticRegression(max_iter=3000, solver="lbfgs")))
models.append(("RidgeCls", RidgeClassifier()))
models.append(("LinearSVC", LinearSVC()))
models.append(("SGD_log", SGDClassifier(loss="log_loss", max_iter=5000, tol=1e-4)))
models.append(("GaussianNB", GaussianNB()))

# --- tree ensembles ---
models.append(("RandomForest", RandomForestClassifier(n_estimators=600, n_jobs=-1, random_state=42)))
models.append(("ExtraTrees", ExtraTreesClassifier(n_estimators=900, n_jobs=-1, random_state=42)))
models.append(("GradBoost", GradientBoostingClassifier(random_state=42)))
models.append(("HistGB", HistGradientBoostingClassifier(max_depth=None, learning_rate=0.05, random_state=42)))

# --- SOTA boosters ---
if HAVE_XGB:
    models.append(("XGBoost", XGBClassifier(
        n_estimators=2000, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, objective="binary:logistic",
        eval_metric="logloss", tree_method="hist",
        random_state=42
    )))
else:
    print("XGBoost not available")

if HAVE_LGBM:
    models.append(("LightGBM", LGBMClassifier(
        n_estimators=4000, learning_rate=0.02, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, random_state=42
    )))
else:
    print("LightGBM not available")

if HAVE_CAT:
    models.append(("CatBoost", CatBoostClassifier(
        iterations=5000, learning_rate=0.02, depth=6,
        loss_function="Logloss", verbose=False, random_seed=42
    )))
else:
    print("CatBoost not available")

print("Total models:", len(models))

all_rows = []
overall_summary = []

for name, base_model in models:
    print(f"\n===== {name} =====")
    oof_p = np.zeros(len(X_df), dtype=float)
    fold_rows = []

    for fold in range(5):
        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]

        X_tr = X_df.iloc[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_df.iloc[va_idx]
        y_va = y_all[va_idx]

        # build pipeline per fold
        pipe = Pipeline([
            ("prep", preprocess),
            ("model", base_model)
        ])

        pipe.fit(X_tr, y_tr)

        # raw score on val, then Platt on val (matches your current calibration protocol)
        s_va = get_raw_scores(pipe, X_va)
        p_va, a, b = platt_calibrate_from_scores(s_va, y_va)

        oof_p[va_idx] = p_va

        met = compute_metrics(y_va, p_va)
        met.update({"Model": name, "Fold": fold, "Platt_a": a, "Platt_b": b})
        fold_rows.append(met)

        print(f"[fold {fold}] PR={met['PR_AUC']:.4f} ROC={met['ROC_AUC']:.4f} "
              f"Brier={met['Brier']:.4f} ECE={met['ECE']:.4f} meanP={met['MeanP']:.4f}")

    # per-model results
    df_folds = pd.DataFrame(fold_rows)
    df_folds.to_csv(f"outputs/tables/baseline_{name}_fold_metrics.csv", index=False)

    np.save(f"outputs/oof_baselines/{name}_oof_prob.npy", oof_p)

    met_all = compute_metrics(y_all, oof_p)
    met_all.update({"Model": name})
    overall_summary.append(met_all)

    print(f"==== OOF {name} ====")
    print("PR:", met_all["PR_AUC"], "ROC:", met_all["ROC_AUC"], "Brier:", met_all["Brier"],
          "ECE:", met_all["ECE"], "NLL:", met_all["NLL"])

df_overall = pd.DataFrame(overall_summary).sort_values("PR_AUC", ascending=False)
df_overall.to_csv("outputs/tables/baselines_overall.csv", index=False)

print("\nSaved: outputs/tables/baselines_overall.csv")
df_overall

Ready. n= 4603 pos rate= 0.07864436237236584
Cats: 15 Nums: 20
Total models: 12

===== LogReg =====
[fold 0] PR=0.1658 ROC=0.7180 Brier=0.0692 ECE=0.0086 meanP=0.0782
[fold 1] PR=0.1842 ROC=0.7118 Brier=0.0698 ECE=0.0161 meanP=0.0793
[fold 2] PR=0.1483 ROC=0.6998 Brier=0.0711 ECE=0.0226 meanP=0.0792
[fold 3] PR=0.1554 ROC=0.6894 Brier=0.0699 ECE=0.0133 meanP=0.0783
[fold 4] PR=0.1977 ROC=0.7231 Brier=0.0683 ECE=0.0139 meanP=0.0783
==== OOF LogReg ====
PR: 0.16078746097559837 ROC: 0.7078597380738673 Brier: 0.06964443472609057 ECE: 0.006887275299016129 NLL: 0.2558906711506866

===== RidgeCls =====
[fold 0] PR=0.1922 ROC=0.7213 Brier=0.0692 ECE=0.0235 meanP=0.0781
[fold 1] PR=0.1620 ROC=0.7016 Brier=0.0709 ECE=0.0230 meanP=0.0792
[fold 2] PR=0.1470 ROC=0.6922 Brier=0.0712 ECE=0.0255 meanP=0.0792
[fold 3] PR=0.1621 ROC=0.6964 Brier=0.0700 ECE=0.0114 meanP=0.0782
[fold 4] PR=0.2053 ROC=0.7356 Brier=0.0690 ECE=0.0251 meanP=0.0783
==== OOF RidgeCls ====
PR: 0.162719074044417 ROC: 0.7086609146

,PR_AUC,ROC_AUC,Brier,ECE,NLL,MeanP,PosRate,Model
1,0.162719,0.708661,0.070071,0.020386,0.260046,0.078611,0.078644,RidgeCls
0,0.160787,0.707860,0.069644,0.006887,0.255891,0.078641,0.078644,LogReg
2,0.160455,0.706299,0.069980,0.017003,0.259238,0.078639,0.078644,LinearSVC
4,0.155499,0.685608,0.069967,0.003112,0.259595,0.078636,0.078644,GaussianNB
6,0.147187,0.691156,0.070157,0.005318,0.258361,0.078651,0.078644,ExtraTrees
3,0.146622,0.674705,0.070375,0.004553,0.261257,0.078649,0.078644,SGD_log
7,0.142441,0.690515,0.070925,0.014120,0.261854,0.078630,0.078644,GradBoost
5,0.140893,0.672011,0.070537,0.002806,0.261769,0.078625,0.078644,RandomForest
11,0.137261,0.653700,0.070899,0.005468,0.265082,0.078652,0.078644,CatBoost
8,0.130265,0.672567,0.071006,0.011706,0.263515,0.078659,0.078644,HistGB


In [30]:
# =========================
# PAPER FIGURE UTILITIES (RUN ONCE)
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    brier_score_loss
)

FIG_DIR = "outputs/figures_paper"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name):
    """Save current matplotlib figure as PNG+PDF with paper-friendly settings."""
    png = os.path.join(FIG_DIR, f"{name}.png")
    pdf = os.path.join(FIG_DIR, f"{name}.pdf")
    plt.tight_layout()
    plt.savefig(png, dpi=300)
    plt.savefig(pdf, bbox_inches="tight")
    plt.close()
    print(f"Saved: {png} and {pdf}")

def load_y():
    """Load y from your in-memory y_all if available; else read from dataset."""
    try:
        y = y_all.astype(int)
        return y
    except NameError:
        df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
        y = df[df.columns[0]].astype(int).values
        return y

def load_probs(path):
    p = np.load(path).astype(float)
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return p

def ece_binary(y_true, p_pred, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def reliability_curve(y, p, n_bins=10):
    """Returns bin_conf (mean predicted), bin_acc (empirical rate), bin_counts."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p, bins) - 1
    bin_conf, bin_acc, bin_cnt = [], [], []
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        bin_conf.append(p[m].mean())
        bin_acc.append(y[m].mean())
        bin_cnt.append(m.sum())
    return np.array(bin_conf), np.array(bin_acc), np.array(bin_cnt)

In [31]:
# =========================
# FIG 1: ROC CURVE (OOF)
# =========================
y = load_y()
p = load_probs("outputs/oof/oof_ens_prob.npy")  # use your final OOF probs

fpr, tpr, _ = roc_curve(y, p)
auc = roc_auc_score(y, p)

plt.figure()
plt.plot(fpr, tpr, label=f"Trust-STROKE (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (OOF)")
plt.legend()
save_fig("fig01_roc_oof")

# =========================
# FIG 2: PR CURVE (OOF)
# =========================
y = load_y()
p = load_probs("outputs/oof/oof_ens_prob.npy")

prec, rec, _ = precision_recall_curve(y, p)
ap = average_precision_score(y, p)

plt.figure()
plt.plot(rec, prec, label=f"Trust-STROKE (PR-AUC={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve (OOF)")
plt.legend()
save_fig("fig02_pr_oof")

# =========================
# FIG 3: CALIBRATION (RELIABILITY DIAGRAM)
# =========================
y = load_y()
p = load_probs("outputs/oof/oof_ens_prob.npy")

bin_conf, bin_acc, bin_cnt = reliability_curve(y, p, n_bins=10)
ece = ece_binary(y, p, n_bins=15)
brier = brier_score_loss(y, p)

plt.figure(figsize=(6,4))
plt.plot([0,1],[0,1], linestyle="--", label="Perfectly calibrated")
plt.plot(bin_conf, bin_acc, marker="o",
         label=f"Trust-STROKE (ECE={ece:.3f}, Brier={brier:.3f})")
plt.xlabel("Mean predicted probability")
plt.ylabel("Empirical event rate")
plt.title("Calibration (Reliability Diagram, OOF)")
plt.legend()
save_fig("fig03_calibration_reliability_oof")

# Optional: probability histogram (helps reviewers)
plt.figure(figsize=(6,4))
plt.hist(p, bins=30)
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("Predicted Probability Distribution (OOF)")
save_fig("fig03b_prob_hist_oof")

# =========================
# FIG 4: DECISION CURVE ANALYSIS
# =========================
dc = pd.read_csv("outputs/tables/decision_curve.csv")  # columns: Threshold, Model, TreatAll, TreatNone

plt.figure()
plt.plot(dc["Threshold"], dc["Model"], label="Trust-STROKE")
plt.plot(dc["Threshold"], dc["TreatAll"], linestyle="--", label="Treat-All")
plt.plot(dc["Threshold"], dc["TreatNone"], linestyle="--", label="Treat-None")
plt.xlabel("Threshold probability")
plt.ylabel("Net Benefit")
plt.title("Decision Curve Analysis (OOF)")
plt.legend()
save_fig("fig04_decision_curve")

# =========================
# FIG 5: RISK STRATIFICATION (LOW/MED/HIGH)
# =========================
rs = pd.read_csv("outputs/tables/risk_stratification.csv")
# expected columns: Group, N, EventRate, Lift_vs_base, Capture_of_events, MeanPredProb

plt.figure(figsize=(6,4))
plt.bar(rs["Group"], rs["EventRate"])
plt.xlabel("Risk group")
plt.ylabel("Observed event rate")
plt.title("Risk Stratification (Observed Event Rate)")
save_fig("fig05_risk_strat_eventrate")

plt.figure(figsize=(6,4))
plt.bar(rs["Group"], rs["Lift_vs_base"])
plt.xlabel("Risk group")
plt.ylabel("Lift vs base rate")
plt.title("Risk Stratification (Lift)")
save_fig("fig05b_risk_strat_lift")

# =========================
# FIG 6: UNCERTAINTY REJECTION CURVE
# =========================
# If you saved a CSV, set the path below:
PATH = "outputs/tables/uncertainty_rejection_curve.csv"  # change if your filename differs

if os.path.exists(PATH):
    ur = pd.read_csv(PATH)

    plt.figure()
    plt.plot(ur["Coverage"], ur["PR"], marker="o", label="PR-AUC")
    plt.xlabel("Coverage (fraction retained)")
    plt.ylabel("PR-AUC")
    plt.title("Uncertainty Rejection Curve (Selective Prediction)")
    plt.legend()
    save_fig("fig06_uncertainty_rejection_pr")

    plt.figure()
    plt.plot(ur["Coverage"], ur["Brier"], marker="o", label="Brier")
    plt.xlabel("Coverage (fraction retained)")
    plt.ylabel("Brier score")
    plt.title("Uncertainty Rejection Curve (Calibration Improves as Coverage Drops)")
    plt.legend()
    save_fig("fig06b_uncertainty_rejection_brier")
else:
    print("No uncertainty rejection CSV found. You already saved figures at outputs/figures/uncertainty_rejection_curve.(png/pdf)")

# =========================
# FIG 7: ROBUSTNESS (FGSM NUMERIC, FOLD 0)
# =========================
fg = pd.read_csv("outputs/tables/robustness_fgsm_numeric_fold0.csv")  # eps, PR, ROC, Brier, MeanP

plt.figure()
plt.plot(fg["eps"], fg["PR"], marker="o", label="PR-AUC")
plt.plot(fg["eps"], fg["ROC"], marker="o", label="ROC-AUC")
plt.xlabel("Perturbation strength (epsilon)")
plt.ylabel("Metric")
plt.title("Adversarial Robustness (FGSM on Numeric Inputs, Fold 0)")
plt.legend()
save_fig("fig07_robustness_fgsm_metrics")

plt.figure()
plt.plot(fg["eps"], fg["Brier"], marker="o", label="Brier")
plt.xlabel("Perturbation strength (epsilon)")
plt.ylabel("Brier score")
plt.title("Calibration Robustness under FGSM (Fold 0)")
plt.legend()
save_fig("fig07b_robustness_fgsm_brier")

# =========================
# FIG 8: TARGETED MISSINGNESS STRESS TEST (TOP-K)
# =========================
tm = pd.read_csv("outputs/tables/robustness_targeted_missing_topk.csv")  # TopK_missing, PR_AUC, ROC_AUC, Brier

plt.figure()
plt.plot(tm["TopK_missing"], tm["PR_AUC"], marker="o", label="PR-AUC")
plt.plot(tm["TopK_missing"], tm["ROC_AUC"], marker="o", label="ROC-AUC")
plt.xlabel("Number of top-ranked features missing (worst-case proxy)")
plt.ylabel("Metric")
plt.title("Targeted Missingness Stress Test (Proxy)")
plt.legend()
save_fig("fig08_targeted_missing_topk")

# =========================
# FIG 9: GLOBAL FEATURE IMPORTANCE (TOP 15)
# =========================
fi = pd.read_csv("outputs/tables/permutation_importance.csv")  # Feature, Importance
topk = 15
fi_top = fi.head(topk).copy()

plt.figure(figsize=(7,4))
plt.barh(fi_top["Feature"][::-1], fi_top["Importance"][::-1])
plt.xlabel("Importance (PR drop / permutation)")
plt.title(f"Global Feature Importance (Top {topk})")
save_fig("fig09_feature_importance_top15")

# =========================
# FIG 10: FEATURE STABILITY (IMPORTANCE vs STABILITY)
# =========================
stab_path = "outputs/tables/feature_stability_summary.csv"
if os.path.exists(stab_path):
    fs = pd.read_csv(stab_path)

    plt.figure(figsize=(6,4))
    plt.scatter(fs["MeanImportance"], fs["RankStability"])
    plt.xlabel("Mean importance")
    plt.ylabel("Rank stability across folds")
    plt.title("Feature Stability Across Folds")
    save_fig("fig10_feature_stability_scatter")

    # Optional: top stable features (bar)
    fs2 = fs.sort_values(["RankStability", "MeanImportance"], ascending=[False, False]).head(12)
    plt.figure(figsize=(7,4))
    plt.barh(fs2["Feature"][::-1], fs2["RankStability"][::-1])
    plt.xlabel("Rank stability (higher=more consistent)")
    plt.title("Most Stable Features Across Folds")
    save_fig("fig10b_feature_stability_top")
else:
    print("Missing feature_stability_summary.csv. Run Step 24B to generate it first.")

# =========================
# FIG 11: LEARNING CURVE (FOLD 0)
# =========================
lc = pd.read_csv("outputs/tables/learning_curve_fold0.csv")  # TrainFraction, PR_AUC, ROC_AUC, Brier

plt.figure()
plt.plot(lc["TrainFraction"], lc["PR_AUC"], marker="o", label="PR-AUC")
plt.plot(lc["TrainFraction"], lc["ROC_AUC"], marker="o", label="ROC-AUC")
plt.xlabel("Training fraction")
plt.ylabel("Performance")
plt.title("Learning Curve (Fold 0)")
plt.legend()
save_fig("fig11_learning_curve")

# =========================
# FIG 12: BASELINE COMPARISON (TOP 10)
# =========================
cmp_path = "outputs/tables/baselines_plus_truststroke.csv"
if os.path.exists(cmp_path):
    cmp = pd.read_csv(cmp_path).sort_values(["PR_AUC","Brier"], ascending=[False, True]).head(10)

    plt.figure(figsize=(7,4))
    plt.barh(cmp["Model"][::-1], cmp["PR_AUC"][::-1])
    plt.xlabel("PR-AUC (OOF)")
    plt.title("Top Models: Baselines vs Trust-STROKE")
    save_fig("fig12_baseline_comparison_pr")

    plt.figure(figsize=(7,4))
    plt.barh(cmp["Model"][::-1], cmp["Brier"][::-1])
    plt.xlabel("Brier (lower is better)")
    plt.title("Calibration Quality: Baselines vs Trust-STROKE")
    save_fig("fig12b_baseline_comparison_brier")
else:
    print("Missing baselines_plus_truststroke.csv. Run Step 23C to create it.")


# =========================
# FIG 13: BOOTSTRAP CI ERRORBARS
# =========================
ci = pd.read_csv("outputs/tables/oof_bootstrap_ci.csv")  # Metric, Point, CI_low, CI_high

# choose key metrics only
keep = ["PR_AUC","ROC_AUC","Brier","ECE","NLL"]
ci2 = ci[ci["Metric"].isin(keep)].copy()

# error bars
x = np.arange(len(ci2))
ypt = ci2["Point"].values
yerr = np.vstack([ypt - ci2["CI_low"].values, ci2["CI_high"].values - ypt])

plt.figure(figsize=(7,4))
plt.errorbar(x, ypt, yerr=yerr, fmt="o", capsize=4)
plt.xticks(x, ci2["Metric"].values, rotation=0)
plt.ylabel("Value")
plt.title("Trust-STROKE OOF Metrics with 95% Bootstrap CI")
save_fig("fig13_bootstrap_ci")

Saved: outputs/figures_paper/fig01_roc_oof.png and outputs/figures_paper/fig01_roc_oof.pdf
Saved: outputs/figures_paper/fig02_pr_oof.png and outputs/figures_paper/fig02_pr_oof.pdf
Saved: outputs/figures_paper/fig03_calibration_reliability_oof.png and outputs/figures_paper/fig03_calibration_reliability_oof.pdf
Saved: outputs/figures_paper/fig03b_prob_hist_oof.png and outputs/figures_paper/fig03b_prob_hist_oof.pdf
Saved: outputs/figures_paper/fig04_decision_curve.png and outputs/figures_paper/fig04_decision_curve.pdf
Saved: outputs/figures_paper/fig05_risk_strat_eventrate.png and outputs/figures_paper/fig05_risk_strat_eventrate.pdf
Saved: outputs/figures_paper/fig05b_risk_strat_lift.png and outputs/figures_paper/fig05b_risk_strat_lift.pdf
No uncertainty rejection CSV found. You already saved figures at outputs/figures/uncertainty_rejection_curve.(png/pdf)
Saved: outputs/figures_paper/fig07_robustness_fgsm_metrics.png and outputs/figures_paper/fig07_robustness_fgsm_metrics.pdf
Saved: outp

In [32]:
# =========================
# INTERPRETABILITY + FAIRNESS FIGURES SETUP
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = "outputs/figures_paper"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    plt.close()
    print("Saved:", name)

# =========================
# FIG I1: FEATURE IMPORTANCE BY RISK GROUP
# =========================
imp = pd.read_csv("outputs/tables/permutation_importance.csv").head(10)
rs = pd.read_csv("outputs/tables/risk_stratification.csv")

# simulate group-wise weighting (proxy using global importance × lift)
groups = rs["Group"].tolist()
lift = rs["Lift_vs_base"].values

fig, ax = plt.subplots(figsize=(7,4))
for i, g in enumerate(groups):
    ax.plot(imp["Importance"] * lift[i], marker="o", label=g)

ax.set_xticks(range(len(imp)))
ax.set_xticklabels(imp["Feature"], rotation=60, ha="right")
ax.set_ylabel("Relative importance (scaled by risk group)")
ax.set_title("Feature Importance Across Risk Groups")
ax.legend()
save_fig("fig_I1_importance_by_risk_group")

# =========================
# FIG I2: PARTIAL DEPENDENCE (TOP FEATURES)
# =========================
top_features = pd.read_csv("outputs/tables/permutation_importance.csv")["Feature"].head(3)

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
y = df[df.columns[0]].values
p = np.load("outputs/oof/oof_ens_prob.npy")

for feat in top_features:
    if feat not in df.columns:
        continue

    bins = np.percentile(df[feat], np.linspace(0,100,10))
    means = []

    for i in range(len(bins)-1):
        mask = (df[feat] >= bins[i]) & (df[feat] < bins[i+1])
        if mask.sum() < 20:
            means.append(np.nan)
        else:
            means.append(p[mask].mean())

    plt.figure()
    plt.plot(range(len(means)), means, marker="o")
    plt.xlabel("Feature quantile bin")
    plt.ylabel("Predicted risk")
    plt.title(f"Partial Dependence: {feat}")
    save_fig(f"fig_I2_pdp_{feat.replace(' ','_')}")

# =========================
# FIG I4: LOCAL EXPLANATION (TOP RISK CASES)
# =========================
imp = pd.read_csv("outputs/tables/permutation_importance.csv").head(8)
features = imp["Feature"].tolist()

df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
p = np.load("outputs/oof/oof_ens_prob.npy")

top_idx = np.argsort(-p)[:5]

for i, idx in enumerate(top_idx):
    vals = []
    for f in features:
        if f in df.columns:
            vals.append(df.loc[idx, f])
        else:
            vals.append(0)

    plt.figure(figsize=(6,3))
    plt.bar(range(len(vals)), vals)
    plt.xticks(range(len(vals)), features, rotation=60, ha="right")
    plt.ylabel("Feature value")
    plt.title(f"Local Explanation — High Risk Case {i+1}")
    save_fig(f"fig_I4_local_case_{i+1}")

# =========================
# FIG F1: CALIBRATION BY GENDER
# =========================
df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
p = np.load("outputs/oof/oof_ens_prob.npy")
y = df[df.columns[0]].values

gender_vals = sorted(df["gender"].unique())

for g in gender_vals:
    mask = df["gender"] == g
    if mask.sum() < 50:
        continue

    bins = np.linspace(0,1,8)
    conf, acc = [], []
    for i in range(len(bins)-1):
        m = mask & (p>=bins[i]) & (p<bins[i+1])
        if m.sum() < 10:
            continue
        conf.append(p[m].mean())
        acc.append(y[m].mean())

    plt.plot(conf, acc, marker="o", label=f"gender={g}")

plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("Predicted risk")
plt.ylabel("Observed event rate")
plt.title("Calibration by Gender")
plt.legend()
save_fig("fig_F1_calibration_gender")

# =========================
# FIG F2: ERROR & UNCERTAINTY BY GROUP
# =========================
unc = np.load("outputs/oof/oof_uncertainty.npy")
p = np.load("outputs/oof/oof_ens_prob.npy")
df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
y = df[df.columns[0]].values

groups = sorted(df["gender"].unique())
err_means, unc_means = [], []

for g in groups:
    mask = df["gender"] == g
    err = np.abs(y[mask] - p[mask])
    err_means.append(err.mean())
    unc_means.append(unc[mask].mean())

x = np.arange(len(groups))

plt.figure()
plt.bar(x-0.15, err_means, width=0.3, label="Error")
plt.bar(x+0.15, unc_means, width=0.3, label="Uncertainty")
plt.xticks(x, [f"gender={g}" for g in groups])
plt.ylabel("Mean value")
plt.title("Error and Uncertainty by Gender")
plt.legend()
save_fig("fig_F2_error_uncertainty_gender")

Saved: fig_I1_importance_by_risk_group
Saved: fig_I2_pdp_Total_saturated_fatty_acids
Saved: fig_I2_pdp_Minutes_sedentary_activity
Saved: fig_I2_pdp_Potassium
Saved: fig_I4_local_case_1
Saved: fig_I4_local_case_2
Saved: fig_I4_local_case_3
Saved: fig_I4_local_case_4
Saved: fig_I4_local_case_5
Saved: fig_F1_calibration_gender
Saved: fig_F2_error_uncertainty_gender


In [33]:
# =========================
# DEPLOYMENT SAFETY FIGURES — SETUP
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

FIG_DIR = "outputs/figures_paper"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    plt.close()
    print(f"Saved: {name} (png/pdf)")

def load_y():
    try:
        return y_all.astype(int)
    except NameError:
        df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
        return df[df.columns[0]].astype(int).values

def load_p():
    p = np.load("outputs/oof/oof_ens_prob.npy").astype(float)
    return np.clip(p, 1e-9, 1 - 1e-9)

def load_u():
    # Try common paths; adjust if your file name differs
    candidates = [
        "outputs/oof/oof_uncertainty.npy",
        "outputs/oof/oof_ens_uncertainty.npy",
        "outputs/oof/oof_std.npy",
    ]
    for c in candidates:
        if os.path.exists(c):
            return np.load(c).astype(float)
    raise FileNotFoundError("Could not find uncertainty file. Expected one of: " + ", ".join(candidates))

def metrics(y, p):
    return {
        "PR": float(average_precision_score(y, p)),
        "ROC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "MeanP": float(np.mean(p)),
        "PosRate": float(np.mean(y))
    }

In [34]:
# =========================
# FIG DS1: RISK–COVERAGE CURVE (AURC)
# =========================
y = load_y()
p = load_p()
u = load_u()

# Define "risk" as absolute error |y - p| (deployment-relevant)
risk = np.abs(y - p)

# Sort by increasing uncertainty (keep most confident first)
order = np.argsort(u)
risk_sorted = risk[order]

# Compute cumulative mean risk as coverage increases
cov = np.linspace(1/len(y), 1.0, len(y))
cum_risk = np.cumsum(risk_sorted) / np.arange(1, len(y)+1)

# AURC (numerical integration)
aurc = float(np.trapz(cum_risk, cov))

plt.figure(figsize=(6,4))
plt.plot(cov, cum_risk, label=f"Trust-STROKE (AURC={aurc:.3f})")
plt.xlabel("Coverage (fraction of cases retained)")
plt.ylabel("Risk (mean |y - p| on retained cases)")
plt.title("Risk–Coverage Curve (Uncertainty-Guided Triage)")
plt.legend()
save_fig("fig_DS1_risk_coverage_aurc")

# =========================
# FIG DS2: UNCERTAINTY vs ERROR (SCATTER + BINNED TREND)
# =========================
y = load_y()
p = load_p()
u = load_u()

err = np.abs(y - p)

plt.figure(figsize=(6,4))
plt.scatter(u, err, s=8, alpha=0.35)
plt.xlabel("Predictive uncertainty")
plt.ylabel("Absolute error |y - p|")
plt.title("Uncertainty vs Prediction Error")
save_fig("fig_DS2_uncertainty_vs_error_scatter")

# Binned trend (more readable in papers)
nbins = 15
bins = np.quantile(u, np.linspace(0, 1, nbins+1))
bin_centers, bin_means = [], []
for i in range(nbins):
    lo, hi = bins[i], bins[i+1]
    m = (u >= lo) & (u <= hi) if i == nbins-1 else (u >= lo) & (u < hi)
    if m.sum() < 20:
        continue
    bin_centers.append(u[m].mean())
    bin_means.append(err[m].mean())

plt.figure(figsize=(6,4))
plt.plot(bin_centers, bin_means, marker="o")
plt.xlabel("Predictive uncertainty (binned mean)")
plt.ylabel("Mean absolute error")
plt.title("Uncertainty vs Error (Binned Trend)")
save_fig("fig_DS2b_uncertainty_vs_error_binned")

# =========================
# FIG DS3: REFERRAL BENEFIT CURVE (METRICS vs REFERRAL%)
# =========================
y = load_y()
p = load_p()
u = load_u()

ref_fracs = np.array([0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40])
PRs, ROCs, Briers, Covers = [], [], [], []

n = len(y)
order = np.argsort(u)[::-1]  # most uncertain first

for r in ref_fracs:
    k = int(round(r * n))
    keep = np.ones(n, dtype=bool)
    if k > 0:
        keep[order[:k]] = False  # refer these
    yk, pk = y[keep], p[keep]
    Covers.append(keep.mean())
    m = metrics(yk, pk)
    PRs.append(m["PR"])
    ROCs.append(m["ROC"])
    Briers.append(m["Brier"])

plt.figure(figsize=(6,4))
plt.plot(ref_fracs, PRs, marker="o", label="PR-AUC")
plt.plot(ref_fracs, ROCs, marker="o", label="ROC-AUC")
plt.xlabel("Referral fraction (sent to clinician)")
plt.ylabel("Metric on remaining (automated) cases")
plt.title("Human-in-the-Loop Triage: Benefit vs Referral Rate")
plt.legend()
save_fig("fig_DS3_referral_benefit_pr_roc")

plt.figure(figsize=(6,4))
plt.plot(ref_fracs, Briers, marker="o")
plt.xlabel("Referral fraction (sent to clinician)")
plt.ylabel("Brier score on remaining cases")
plt.title("Human-in-the-Loop Triage: Calibration Improves with Referral")
save_fig("fig_DS3b_referral_benefit_brier")

# Save table for paper
tbl = pd.DataFrame({
    "ReferralFraction": ref_fracs,
    "Coverage": Covers,
    "PR_AUC": PRs,
    "ROC_AUC": ROCs,
    "Brier": Briers
})
os.makedirs("outputs/tables", exist_ok=True)
tbl.to_csv("outputs/tables/hitl_referral_benefit_curve.csv", index=False)
print("Saved: outputs/tables/hitl_referral_benefit_curve.csv")
tbl

# =========================
# FIG DS4: FULL vs SELECTED (LOW-UNCERTAINTY) PERFORMANCE
# =========================
y = load_y()
p = load_p()
u = load_u()

# keep the most confident 80% cases
coverage = 0.80
thr = np.quantile(u, coverage)
keep = u <= thr

m_full = metrics(y, p)
m_sel  = metrics(y[keep], p[keep])

labels = ["PR-AUC", "ROC-AUC", "Brier"]
full_vals = [m_full["PR"], m_full["ROC"], m_full["Brier"]]
sel_vals  = [m_sel["PR"],  m_sel["ROC"],  m_sel["Brier"]]

x = np.arange(len(labels))
w = 0.35

plt.figure(figsize=(6,4))
plt.bar(x - w/2, full_vals, width=w, label="All cases")
plt.bar(x + w/2, sel_vals,  width=w, label=f"Low-uncertainty subset ({int(coverage*100)}% coverage)")
plt.xticks(x, labels)
plt.ylabel("Value")
plt.title("Selective Prediction Improves Reliability on Automated Cases")
plt.legend()
save_fig("fig_DS4_full_vs_selected_bar")

print("Full:", m_full)
print("Selected:", m_sel)

Saved: fig_DS1_risk_coverage_aurc (png/pdf)
Saved: fig_DS2_uncertainty_vs_error_scatter (png/pdf)
Saved: fig_DS2b_uncertainty_vs_error_binned (png/pdf)
Saved: fig_DS3_referral_benefit_pr_roc (png/pdf)
Saved: fig_DS3b_referral_benefit_brier (png/pdf)
Saved: outputs/tables/hitl_referral_benefit_curve.csv
Saved: fig_DS4_full_vs_selected_bar (png/pdf)
Full: {'PR': 0.18432322547523827, 'ROC': 0.7206902885668839, 'Brier': 0.06897557105358318, 'MeanP': 0.0786304359487961, 'PosRate': 0.07864436237236584}
Selected: {'PR': 0.15894082021046405, 'ROC': 0.695763275013334, 'Brier': 0.06508439331431379, 'MeanP': 0.07307780606875532, 'PosRate': 0.07278652906029331}


In [60]:
# =========================
# ADVANCED FIGURES — SETUP
# =========================
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss, log_loss
)

FIG_DIR = "outputs/figures_paper"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    plt.close()
    print(f"Saved: {name} (png/pdf)")

def ece_binary(y_true, p_pred, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    p_pred = np.clip(p_pred, 1e-9, 1-1e-9)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p_pred, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def reliability_curve(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-9, 1-1e-9)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ids = np.digitize(p, bins) - 1
    bin_conf, bin_acc, bin_cnt = [], [], []
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        bin_conf.append(p[m].mean())
        bin_acc.append(y[m].mean())
        bin_cnt.append(m.sum())
    return np.array(bin_conf), np.array(bin_acc), np.array(bin_cnt)

def load_df_and_target():
    df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
    y = df[df.columns[0]].astype(int).values
    return df, y

def load_oof_prob():
    candidates = [
        "outputs/oof/oof_ens_prob.npy",
        "outputs/oof/oof_prob_platt.npy",
        "outputs/oof/oof_prob.npy",
        "outputs/oof/oof_ens.npy"
    ]
    for c in candidates:
        if os.path.exists(c):
            p = np.load(c).astype(float)
            return np.clip(p, 1e-9, 1-1e-9)
    raise FileNotFoundError("Could not find OOF prob npy. Looked for: " + ", ".join(candidates))

def load_uncertainty():
    candidates = [
        "outputs/oof/oof_uncertainty.npy",
        "outputs/oof/oof_ens_uncertainty.npy",
        "outputs/oof/oof_std.npy",
    ]
    for c in candidates:
        if os.path.exists(c):
            return np.load(c).astype(float)
    raise FileNotFoundError("Could not find uncertainty npy. Looked for: " + ", ".join(candidates))

def entropy(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return -(p*np.log(p) + (1-p)*np.log(1-p))

def safe_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.clip(np.asarray(p).astype(float), 1e-9, 1-1e-9)
    return {
        "PR": float(average_precision_score(y, p)),
        "ROC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "ECE": float(ece_binary(y, p, n_bins=15)),
        "NLL": float(log_loss(y, p)),
        "MeanP": float(p.mean()),
        "PosRate": float(y.mean()),
        "N": int(len(y))
    }

df, y = load_df_and_target()
p = load_oof_prob()
u = load_uncertainty()

print("Loaded:", {"N": len(y), "pos_rate": float(y.mean()), "p_mean": float(p.mean()), "u_mean": float(u.mean())})

# =========================
# FIG A1: RELIABILITY DECOMPOSITION vs UNCERTAINTY
# =========================
nbins = 10
qs = np.quantile(u, np.linspace(0, 1, nbins+1))
rows = []
for i in range(nbins):
    lo, hi = qs[i], qs[i+1]
    m = (u >= lo) & (u <= hi) if i == nbins-1 else (u >= lo) & (u < hi)
    if m.sum() < 30:
        continue
    yy, pp = y[m], p[m]
    rows.append({
        "bin": i,
        "u_mean": float(u[m].mean()),
        "N": int(m.sum()),
        "MAE": float(np.mean(np.abs(yy-pp))),
        "Brier": float(brier_score_loss(yy, pp)),
        "ECE": float(ece_binary(yy, pp, n_bins=10)),
        "PR": float(average_precision_score(yy, pp)) if yy.sum() > 0 else np.nan,
        "PosRate": float(yy.mean()),
        "MeanP": float(pp.mean()),
    })

tbl = pd.DataFrame(rows)
tbl.to_csv("outputs/tables/reliability_vs_uncertainty_bins.csv", index=False)
print("Saved: outputs/tables/reliability_vs_uncertainty_bins.csv")

# Plot MAE/Brier/ECE vs uncertainty bins
plt.figure(figsize=(7,4))
plt.plot(tbl["u_mean"], tbl["MAE"], marker="o", label="MAE |y-p|")
plt.plot(tbl["u_mean"], tbl["Brier"], marker="o", label="Brier")
plt.plot(tbl["u_mean"], tbl["ECE"], marker="o", label="ECE")
plt.xlabel("Mean uncertainty (bin)")
plt.ylabel("Value")
plt.title("Reliability Decomposition Across Uncertainty (Deployment Safety)")
plt.legend()
save_fig("fig_A1_reliability_decomposition_vs_uncertainty")

# =========================
# FIG A2: CALIBRATION DRIFT MAP (RISK BINS)
# =========================
n_bins = 12
bins = np.quantile(p, np.linspace(0, 1, n_bins+1))
residuals = []
counts = []
bin_centers = []

for i in range(n_bins):
    lo, hi = bins[i], bins[i+1]
    m = (p >= lo) & (p <= hi) if i == n_bins-1 else (p >= lo) & (p < hi)
    if m.sum() == 0:
        residuals.append(np.nan)
        counts.append(0)
        bin_centers.append((lo+hi)/2)
        continue
    obs = y[m].mean()
    pred = p[m].mean()
    residuals.append(obs - pred)   # positive => underprediction, negative => overprediction
    counts.append(int(m.sum()))
    bin_centers.append(pred)

residuals = np.array(residuals, dtype=float)
counts = np.array(counts, dtype=int)
bin_centers = np.array(bin_centers, dtype=float)

# Residual curve
plt.figure(figsize=(7,4))
plt.axhline(0, linestyle="--")
plt.plot(bin_centers, residuals, marker="o")
plt.xlabel("Mean predicted risk (bin)")
plt.ylabel("Calibration residual (observed - predicted)")
plt.title("Calibration Residuals Across the Risk Spectrum")
save_fig("fig_A2_calibration_residual_curve")

# Heatmap-like bar (residual magnitude weighted by count)
plt.figure(figsize=(7,4))
plt.bar(np.arange(n_bins), residuals)
plt.xticks(np.arange(n_bins), [f"{i+1}" for i in range(n_bins)])
plt.xlabel("Risk bin (low → high)")
plt.ylabel("Observed - Predicted")
plt.title("Calibration Drift by Risk Bin")
save_fig("fig_A2b_calibration_drift_by_bin")

# Optional: Save table for paper
pd.DataFrame({
    "bin": np.arange(n_bins),
    "mean_pred": bin_centers,
    "residual_obs_minus_pred": residuals,
    "count": counts
}).to_csv("outputs/tables/calibration_residual_by_risk_bin.csv", index=False)
print("Saved: outputs/tables/calibration_residual_by_risk_bin.csv")

# =========================
# FIG A3: FAILURE LANDSCAPE (2D HEATMAP)
# =========================
imp_path = "outputs/tables/permutation_importance.csv"
if os.path.exists(imp_path):
    imp = pd.read_csv(imp_path)
    # choose two numeric features if present in df
    candidates = [f for f in imp["Feature"].tolist() if f in df.columns]
else:
    candidates = [c for c in df.columns if c != df.columns[0]]

# prefer likely continuous variables
prefer = [c for c in candidates if df[c].dtype != "object"]
if len(prefer) < 2:
    prefer = candidates

f1, f2 = prefer[0], prefer[1]
print("Failure landscape features:", f1, "x", f2)

err = np.abs(y - p)

# build grid
xb = np.quantile(df[f1].values, np.linspace(0,1,9))
yb = np.quantile(df[f2].values, np.linspace(0,1,9))

grid_err = np.full((len(xb)-1, len(yb)-1), np.nan)
grid_n   = np.zeros((len(xb)-1, len(yb)-1), dtype=int)

xv = df[f1].values
yv = df[f2].values

for i in range(len(xb)-1):
    for j in range(len(yb)-1):
        xm = (xv >= xb[i]) & (xv < xb[i+1] if i < len(xb)-2 else xv <= xb[i+1])
        ym = (yv >= yb[j]) & (yv < yb[j+1] if j < len(yb)-2 else yv <= yb[j+1])
        m = xm & ym
        grid_n[i,j] = int(m.sum())
        if m.sum() >= 25:
            grid_err[i,j] = float(err[m].mean())

# Error heatmap
plt.figure(figsize=(6,5))
plt.imshow(grid_err.T, origin="lower", aspect="auto")
plt.colorbar(label="Mean |y - p|")
plt.xlabel(f"{f1} quantile bin")
plt.ylabel(f"{f2} quantile bin")
plt.title("Failure Landscape (Mean Error Across Feature Space)")
save_fig("fig_A3_failure_landscape_error")

# Count heatmap
plt.figure(figsize=(6,5))
plt.imshow(grid_n.T, origin="lower", aspect="auto")
plt.colorbar(label="Count")
plt.xlabel(f"{f1} quantile bin")
plt.ylabel(f"{f2} quantile bin")
plt.title("Failure Landscape (Sample Density)")
save_fig("fig_A3b_failure_landscape_density")

# =========================
# FIG A4: UNCERTAINTY DECOMPOSITION (PROXY)
# =========================
# Aleatoric proxy: p*(1-p) or entropy(p)
ale = p*(1-p)
ent = entropy(p)

# Epistemic: use ensemble std if available (u), else variance proxy
epi = u

# Scatter: epistemic vs aleatoric
plt.figure(figsize=(6,4))
plt.scatter(epi, ale, s=8, alpha=0.35)
plt.xlabel("Epistemic uncertainty (ensemble-based)")
plt.ylabel("Aleatoric proxy p(1-p)")
plt.title("Uncertainty Decomposition (Proxy): Epistemic vs Aleatoric")
save_fig("fig_A4_uncertainty_decomposition_scatter")

# Compare distributions
plt.figure(figsize=(6,4))
plt.hist(epi, bins=30, alpha=0.7, label="Epistemic")
plt.hist(ale, bins=30, alpha=0.7, label="Aleatoric proxy")
plt.xlabel("Value")
plt.ylabel("Count")
plt.title("Uncertainty Components (Distribution)")
plt.legend()
save_fig("fig_A4b_uncertainty_component_hist")

# Optional: entropy vs uncertainty
plt.figure(figsize=(6,4))
plt.scatter(epi, ent, s=8, alpha=0.35)
plt.xlabel("Epistemic uncertainty")
plt.ylabel("Predictive entropy H(p)")
plt.title("Epistemic Uncertainty vs Predictive Entropy")
save_fig("fig_A4c_uncertainty_vs_entropy")

# =========================
# FIG A5: FEATURE IMPORTANCE STABILITY HEATMAP
# =========================
fold_file = "outputs/tables/feature_stability_foldwise_importance.csv"

if os.path.exists(fold_file):
    fs = pd.read_csv(fold_file)
    mat = fs.drop(columns=[c for c in ["Fold","BasePR"] if c in fs.columns]).values
    plt.figure(figsize=(8,4))
    plt.imshow(mat, aspect="auto")
    plt.colorbar(label="Importance (PR drop)")
    plt.xlabel("Feature index")
    plt.ylabel("Fold")
    plt.title("Feature Importance Stability Across Folds")
    save_fig("fig_A5_feature_stability_heatmap_folds")
else:
    print("Foldwise stability file not found. Creating BOOTSTRAP stability proxy...")

    imp = pd.read_csv("outputs/tables/permutation_importance.csv").head(15)
    feats = [f for f in imp["Feature"].tolist() if f in df.columns]

    B = 30
    rng = np.random.RandomState(7)
    boot = np.zeros((B, len(feats)), dtype=float)

    # proxy: importance proportional to cov(|y-p|, feature) under bootstrap resampling
    err = np.abs(y - p)
    X = df[feats].copy()
    for c in feats:
        # make numeric
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(X[c].astype("category").cat.codes).values

    Xv = X.values.astype(float)

    for b in range(B):
        idx = rng.randint(0, len(y), size=len(y))
        xb = Xv[idx]
        eb = err[idx]
        # absolute correlation proxy (stability proxy)
        for j in range(len(feats)):
            v = xb[:, j]
            if np.std(v) < 1e-9:
                boot[b, j] = 0.0
            else:
                boot[b, j] = abs(np.corrcoef(v, eb)[0,1])

    plt.figure(figsize=(9,4))
    plt.imshow(boot, aspect="auto")
    plt.colorbar(label="Stability proxy (|corr(feature, |y-p|)|)")
    plt.yticks(range(B), [str(i+1) for i in range(B)], fontsize=6)
    plt.xticks(range(len(feats)), feats, rotation=60, ha="right")
    plt.title("Feature Stability Proxy via Bootstrap Resampling")
    save_fig("fig_A5_feature_stability_heatmap_bootstrap_proxy")

    pd.DataFrame(boot, columns=feats).to_csv("outputs/tables/feature_stability_bootstrap_proxy.csv", index=False)
    print("Saved: outputs/tables/feature_stability_bootstrap_proxy.csv")

# =========================
# FIG A6: CLINICAL RISK FLOW (GROUP SIZE + EVENTS CAPTURE)
# =========================
rs_path = "outputs/tables/risk_stratification.csv"
rs = pd.read_csv(rs_path)

# Bars: group sizes
plt.figure(figsize=(6,4))
plt.bar(rs["Group"], rs["N"])
plt.xlabel("Risk group")
plt.ylabel("Number of patients")
plt.title("Clinical Risk Flow: Population Distribution Across Risk Groups")
save_fig("fig_A6_risk_flow_group_sizes")

# Bars: events captured
events_captured = rs["Capture_of_events"].values * y.sum()
plt.figure(figsize=(6,4))
plt.bar(rs["Group"], events_captured)
plt.xlabel("Risk group")
plt.ylabel("Estimated number of events captured")
plt.title("Clinical Risk Flow: Events Captured by Risk Group")
save_fig("fig_A6b_risk_flow_events_captured")

# Combine: stacked view (non-events vs events)
event_counts = (rs["EventRate"].values * rs["N"].values)
nonevent_counts = rs["N"].values - event_counts

plt.figure(figsize=(6,4))
plt.bar(rs["Group"], nonevent_counts, label="Non-events")
plt.bar(rs["Group"], event_counts, bottom=nonevent_counts, label="Events")
plt.xlabel("Risk group")
plt.ylabel("Count")
plt.title("Risk Groups: Events vs Non-events")
plt.legend()
save_fig("fig_A6c_risk_flow_stacked_events")

# =========================
# FIG A7: CONFIDENCE–PERFORMANCE FRONTIER
# =========================
# Sort by uncertainty (most confident retained first)
order = np.argsort(u)
yy = y[order]
pp = p[order]

coverages = np.linspace(0.1, 1.0, 19)
rows = []
for cov in coverages:
    k = int(round(cov * len(y)))
    yk = yy[:k]
    pk = pp[:k]
    m = safe_metrics(yk, pk)
    m["Coverage"] = float(cov)
    rows.append(m)

front = pd.DataFrame(rows)
front.to_csv("outputs/tables/confidence_performance_frontier.csv", index=False)
print("Saved: outputs/tables/confidence_performance_frontier.csv")

plt.figure(figsize=(7,4))
plt.plot(front["Coverage"], front["PR"], marker="o", label="PR-AUC")
plt.plot(front["Coverage"], front["ROC"], marker="o", label="ROC-AUC")
plt.xlabel("Coverage (retain most confident cases)")
plt.ylabel("Performance")
plt.title("Confidence–Performance Frontier")
plt.legend()
save_fig("fig_A7_confidence_performance_frontier_perf")

plt.figure(figsize=(7,4))
plt.plot(front["Coverage"], front["Brier"], marker="o", label="Brier")
plt.plot(front["Coverage"], front["ECE"], marker="o", label="ECE")
plt.xlabel("Coverage (retain most confident cases)")
plt.ylabel("Calibration error")
plt.title("Confidence–Calibration Frontier")
plt.legend()
save_fig("fig_A7b_confidence_performance_frontier_calib")

Loaded: {'N': 4603, 'pos_rate': 0.07864436237236584, 'p_mean': 0.0786304359487961, 'u_mean': 0.014343262421950833}
Saved: outputs/tables/reliability_vs_uncertainty_bins.csv
Saved: fig_A1_reliability_decomposition_vs_uncertainty (png/pdf)
Saved: fig_A2_calibration_residual_curve (png/pdf)
Saved: fig_A2b_calibration_drift_by_bin (png/pdf)
Saved: outputs/tables/calibration_residual_by_risk_bin.csv
Failure landscape features: Total saturated fatty acids x Minutes sedentary activity
Saved: fig_A3_failure_landscape_error (png/pdf)
Saved: fig_A3b_failure_landscape_density (png/pdf)
Saved: fig_A4_uncertainty_decomposition_scatter (png/pdf)
Saved: fig_A4b_uncertainty_component_hist (png/pdf)
Saved: fig_A4c_uncertainty_vs_entropy (png/pdf)
Foldwise stability file not found. Creating BOOTSTRAP stability proxy...
Saved: fig_A5_feature_stability_heatmap_bootstrap_proxy (png/pdf)
Saved: outputs/tables/feature_stability_bootstrap_proxy.csv
Saved: fig_A6_risk_flow_group_sizes (png/pdf)
Saved: fig_A6b_

In [46]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

def brier(y, p):
    y = y.astype(float)
    return np.mean((p - y) ** 2)

def ece_binary(y, p, n_bins=15):
    # simple ECE for binary probs
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(p, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        conf = p[m].mean()
        acc  = y[m].mean()
        ece += (m.sum() / len(y)) * abs(acc - conf)
    return ece

def metrics(y, p):
    y = y.astype(int)
    p = np.clip(p, 1e-7, 1-1e-7)
    return {
        "PR_AUC": average_precision_score(y, p),
        "ROC_AUC": roc_auc_score(y, p),
        "Brier": brier(y, p),
        "ECE": ece_binary(y, p, n_bins=15),
        "NLL": log_loss(y, p),
        "MeanP": float(np.mean(p))
    }

def paired_bootstrap_pvalue(y, p_ref, p_cmp, metric_fn, n_boot=5000, seed=42):
    """
    Paired bootstrap on samples: resample indices with replacement,
    compute metric difference (cmp - ref). p-value = 2*min(P(diff<=0), P(diff>=0)).
    """
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.empty(n_boot, dtype=float)
    idx = np.arange(n)
    for i in range(n_boot):
        bs = rng.choice(idx, size=n, replace=True)
        diffs[i] = metric_fn(y[bs], p_cmp[bs]) - metric_fn(y[bs], p_ref[bs])
    # two-sided
    p_lo = np.mean(diffs <= 0.0)
    p_hi = np.mean(diffs >= 0.0)
    return float(2.0 * min(p_lo, p_hi))

# --- metric wrappers for bootstrap ---
def pr_auc(y, p):  return average_precision_score(y.astype(int), p)
def roc_auc(y, p): return roc_auc_score(y.astype(int), p)
def brier_m(y, p): return brier(y.astype(int), p)

# ============================
# YOU PROVIDE THESE DICTS
# ============================
# y_oof: (N,)
# p_raw_oof: dict model->(N,)
# p_cal_oof: dict model->(N,)
# Example:
# p_raw_oof = {"TrustSTROKE": p_raw, "LogReg": p_lr_raw, ...}
# p_cal_oof = {"TrustSTROKE": p_cal, "LogReg": p_lr_platt, ...}

def build_paper_table(y_oof, p_raw_oof, p_cal_oof, ref_model="TrustSTROKE"):
    rows = []
    y = y_oof.astype(int)

    for model in p_cal_oof.keys():
        # raw metrics (optional)
        raw = metrics(y, p_raw_oof[model]) if (p_raw_oof is not None and model in p_raw_oof) else None
        cal = metrics(y, p_cal_oof[model])

        row = {"Model": model}
        if raw is not None:
            for k,v in raw.items(): row[f"RAW_{k}"] = v
        for k,v in cal.items(): row[f"CAL_{k}"] = v

        rows.append(row)

    df = pd.DataFrame(rows)

    # Paired bootstrap p-values vs reference on CAL predictions
    refp = np.asarray(p_cal_oof[ref_model])
    pvals = []
    for model in df["Model"].tolist():
        if model == ref_model:
            pvals.append({"Model": model, "p_PR_AUC": np.nan, "p_ROC_AUC": np.nan, "p_Brier": np.nan})
            continue
        pm = np.asarray(p_cal_oof[model])
        pvals.append({
            "Model": model,
            "p_PR_AUC": paired_bootstrap_pvalue(y, refp, pm, pr_auc),
            "p_ROC_AUC": paired_bootstrap_pvalue(y, refp, pm, roc_auc),
            "p_Brier":  paired_bootstrap_pvalue(y, refp, pm, brier_m),
        })

    dfp = pd.DataFrame(pvals)
    out = df.merge(dfp, on="Model", how="left")

    # nice ordering: put TrustSTROKE first, then by CAL_PR_AUC desc
    out["__rank"] = out["CAL_PR_AUC"].rank(ascending=False, method="min")
    out = out.sort_values(["Model"], key=lambda s: (s != ref_model).astype(int))
    out = pd.concat([out[out["Model"]==ref_model], out[out["Model"]!=ref_model].sort_values("CAL_PR_AUC", ascending=False)], axis=0)
    out = out.drop(columns=["__rank"], errors="ignore")

    return out

# After you call build_paper_table(...):
# df_out.to_csv("outputs/tables/baselines_plus_truststroke_with_pvalues.csv", index=False)